# M21C LS Unified Paper Figures

This notebook is the paper-figure control room. It starts with shared paths, period definitions, and input-file availability checks only. Plotting sections should be added after these checks are clean enough for the figure being rebuilt.

In [ ]:
from pathlib import Path
import os
import sys

os.environ.setdefault("MKL_THREADING_LAYER", "SEQUENTIAL")
os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("MKL_NUM_THREADS", "1")
os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")
os.environ.setdefault("NUMEXPR_NUM_THREADS", "1")

import numpy as np
import pandas as pd


def find_repo_root(start: Path) -> Path:
    for p in [start.resolve(), *start.resolve().parents]:
        if (p / ".git").exists() and (p / "common/python/io/read_GEOSldas.py").exists():
            return p
    raise FileNotFoundError("Could not locate geosldas-analysis repo root")


HERE = Path.cwd().resolve()
REPO_ROOT = find_repo_root(HERE)
PROJECT_ROOT = REPO_ROOT / "projects/M21C_ls"
SCRIPTS_ROOT = PROJECT_ROOT / "scripts"
if str(SCRIPTS_ROOT) not in sys.path:
    sys.path.insert(0, str(SCRIPTS_ROOT))
PAPER_FIG_DIR = PROJECT_ROOT / "output/paper_figures"
PAPER_FIG_DIR.mkdir(parents=True, exist_ok=True)

GEOSLDAS_DIAG_ROOT = Path("/Users/amfox/Desktop/GEOSldas_diagnostics/test_data/M21C_land_sweeper_v2")
DISCOVER_REPO_ROOT = Path("/discover/nobackup/projects/land_da/geosldas-analysis")

print("REPO_ROOT:", REPO_ROOT)
print("PAPER_FIG_DIR:", PAPER_FIG_DIR)
print("GEOSLDAS_DIAG_ROOT exists:", GEOSLDAS_DIAG_ROOT.exists())

## Period Registry

Load the shared observing-system registry from `projects/M21C_ls/config/observing_system_registry.json`. Fine periods drive timeline/time-series/OFA figures; broader validation periods drive ISMN and ERA5-Land summaries. Segment-reliability fields carry the P7 short-period constraint into downstream analyses.

In [ ]:
from m21c_periods import load_period_frames

period_registry, fine_periods, validation_periods, sensor_rows = load_period_frames()
PAPER_START = pd.Timestamp(period_registry["paper_start"])
PAPER_END = pd.Timestamp(period_registry["paper_end"])

display(fine_periods)
display(validation_periods)

## Figure And Data Registry

`required` files are needed to rebuild the figure from data. `reference` files are existing figure products useful for visual comparison.

In [ ]:
def p_rel(path: str) -> Path:
    return REPO_ROOT / path


def p_diag(path: str) -> Path:
    return GEOSLDAS_DIAG_ROOT / path


def p_discover(path: str) -> Path:
    return DISCOVER_REPO_ROOT / path


figure_registry = [
    {
        "figure": "Fig. 1",
        "description": "Observing-system timeline",
        "period_set": "fine_periods",
        "required": [],
        "reference": [],
        "upstream": "Period registry in this notebook",
    },
    {
        "figure": "Fig. 2",
        "description": "Mean assimilated observations per day",
        "period_set": "full record",
        "required": [
            p_diag("temporal_stats_DA_20000601_20240531.nc4"),
            p_diag("spatial_stats_DA_200006_202405.pkl"),
            p_diag("LS_OLv8_M36.ldas_tilecoord.bin"),
        ],
        "reference": [],
        "upstream": "projects/M21C_ls/notebooks/LS_ofa_figures_refactor_20260327.ipynb",
    },
    {
        "figure": "Fig. 3",
        "description": "Monthly assimilated observation counts",
        "period_set": "fine_periods",
        "required": [p_diag("spatial_stats_DA_200006_202405.pkl")],
        "reference": [],
        "upstream": "projects/M21C_ls/notebooks/LS_ofa_figures_refactor_20260327.ipynb",
    },
    {
        "figure": "Fig. 4",
        "description": "Full-period OmF maps by sensor",
        "period_set": "full record",
        "required": [
            p_diag("temporal_stats_OL_20000601_20240531.nc4"),
            p_diag("temporal_stats_DA_20000601_20240531.nc4"),
            p_diag("LS_OLv8_M36.ldas_tilecoord.bin"),
        ],
        "reference": [],
        "upstream": "projects/M21C_ls/notebooks/LS_ofa_figures_refactor_20260327.ipynb",
    },
    {
        "figure": "Fig. 5",
        "description": "Monthly normalized OmF evolution",
        "period_set": "fine_periods",
        "required": [
            p_diag("spatial_stats_OL_200006_202405.pkl"),
            p_diag("spatial_stats_DA_200006_202405.pkl"),
        ],
        "reference": [],
        "upstream": "projects/M21C_ls/notebooks/LS_ofa_figures_refactor_20260327.ipynb",
    },
    {
        "figure": "Fig. 6",
        "description": "OmF maps by period and sensor",
        "period_set": "fine_periods where available; P1/P2 need new temporal stats if plotted separately",
        "required": [p_diag("LS_OLv8_M36.ldas_tilecoord.bin")],
        "reference": [],
        "upstream": "projects/M21C_ls/notebooks/LS_ofa_figures_refactor_20260327.ipynb",
    },
    {
        "figure": "Fig. 7",
        "description": "ISMN surface/root-zone soil moisture skill deltas",
        "period_set": "validation_periods",
        "required": [p_rel("projects/M21C_ls/output/ismn_network_skill/batch_figures/all_networks_hybrid_OL_DA_delta_surface_rz_R_anomR_ubRMSE_table.csv")],
        "reference": [p_rel("projects/M21C_ls/output/ismn_network_skill/batch_figures/all_networks_hybrid_OL_DA_delta_surface_rz_R_anomR_ubRMSE.png")],
        "upstream": "projects/M21C_ls/notebooks/insitu_skill_cached_batch_figures.ipynb",
    },
    {
        "figure": "Fig. 8",
        "description": "IMS snow-cover categorical skill deltas and Terra/Aqua scope supplement",
        "period_set": "full record",
        "required": [
            p_rel("projects/IMS/output/ims_ol_da_cell_counts_metrics_SMAP_EASEv2_M36_GLOBAL_2000_2024_thr0p50_imsSnowDaysGe10.nc4"),
            p_rel("projects/IMS/output/ims_ol_da_comparison_table_SMAP_EASEv2_M36_GLOBAL_2000_2024_thr0p50_imsSnowDaysGe10.csv"),
            p_rel("projects/IMS/output/ims_ol_da_scope_metadata_SMAP_EASEv2_M36_GLOBAL_2000_2024_thr0p50_imsSnowDaysGe10.csv"),
            p_rel("projects/IMS/output/ims_ol_da_cell_counts_metrics_SMAP_EASEv2_M36_GLOBAL_2000_2007_thr0p50_imsSnowDaysGe10_terraAquaScopes.nc4"),
            p_rel("projects/IMS/output/ims_ol_da_comparison_table_SMAP_EASEv2_M36_GLOBAL_2000_2007_thr0p50_imsSnowDaysGe10_terraAquaScopes.csv"),
            p_rel("projects/IMS/output/ims_ol_da_scope_metadata_SMAP_EASEv2_M36_GLOBAL_2000_2007_thr0p50_imsSnowDaysGe10_terraAquaScopes.csv"),
        ],
        "reference": [p_rel("projects/IMS/output/figures_ims_maps_and_tables/ims_all_period_delta_metrics_SMAP_EASEv2_M36_GLOBAL_2000_2024_thr0p50_imsSnowDaysGe10_nh_robinson_2x3.png")],
        "upstream": "projects/IMS/scripts/run_ims_ol_da_cell_metrics.py; projects/IMS/notebooks/ims_maps_and_tables_from_precomputed_outputs.ipynb",
    },
    {
        "figure": "Fig. 9",
        "description": "SNOTEL SWE validation",
        "period_set": "full record/seasons",
        "required": [
            p_rel("projects/SNOTEL/outputs_snotel_ol_da_validation/snotel_station_metrics_SMAP_EASEv2_M36_GLOBAL_20000601_20240601.csv"),
            p_rel("projects/SNOTEL/outputs_snotel_ol_da_validation/tables/snotel_swe_toprow_bar_values_ci_SMAP_EASEv2_M36_GLOBAL_20000601_20240601.csv"),
        ],
        "reference": [p_rel("projects/SNOTEL/outputs_snotel_ol_da_validation/figures/snotel_swe_2x3_bars_allsites_maps_da_minus_ol_elevfilt500_SMAP_EASEv2_M36_GLOBAL_20000601_20240601.png")],
        "upstream": "projects/SNOTEL/notebooks/snow_daily_seasonal_ol_da_swe_snwd.ipynb",
    },
    {
        "figure": "Fig. 10",
        "description": "GHCN snow-depth validation",
        "period_set": "full record/seasons",
        "required": [p_rel("projects/GHCN_snwd/outputs_ghcn_snwd_ol_da_validation/ghcn_station_metrics_baseline_core_SMAP_EASEv2_M36_GLOBAL_20000101_20241231.csv")],
        "reference": [p_rel("projects/GHCN_snwd/outputs_ghcn_snwd_ol_da_validation/figures/ghcn_baseline_core_snodpland_2x3_bars_maps_nh_da_minus_ol_ALL_SMAP_EASEv2_M36_GLOBAL_20000101_20241231.png")],
        "upstream": "projects/GHCN_snwd/notebooks/ghcn_snwd_daily_seasonal_ol_da_snwd_baseline_basic.ipynb",
    },
    {
        "figure": "Fig. 11",
        "description": "ERA5-Land soil-moisture bars",
        "period_set": "validation_periods",
        "required": [
            p_rel("projects/era5_land/notebooks/ERA5L_vs_OLv8_M36_strict_summary.nc"),
            p_rel("projects/era5_land/notebooks/ERA5L_vs_DAv8_M36_strict_summary.nc"),
        ],
        "reference": [p_rel("projects/era5_land/notebooks/figures_era5l_bars/bars_surface_rz_sm_combined_3x3_era5l_only.png")],
        "upstream": "projects/era5_land/notebooks/plot_ERA5L_comparison_bars.ipynb",
    },
    {
        "figure": "Fig. 12",
        "description": "ERA5-Land soil-moisture maps",
        "period_set": "validation_periods",
        "required": [
            p_rel("projects/era5_land/notebooks/ERA5L_vs_OLv8_M36_strict_summary.nc"),
            p_rel("projects/era5_land/notebooks/ERA5L_vs_DAv8_M36_strict_summary.nc"),
            p_diag("LS_OLv8_M36.ldas_tilecoord.bin"),
        ],
        "reference": [p_rel("projects/era5_land/notebooks/figures_era5l_postage_stamps/postage_stamp_da_minus_ol_surface_rz_sm_6x3.png")],
        "upstream": "projects/era5_land/notebooks/plot_ERA5L_postage_stamp_maps.ipynb",
    },
    {
        "figure": "Fig. 13",
        "description": "ERA5-Land snow comparison",
        "period_set": "full record",
        "required": [
            p_rel("projects/era5_land/notebooks/ERA5L_vs_OLv8_M36_strict_summary.nc"),
            p_rel("projects/era5_land/notebooks/ERA5L_vs_DAv8_M36_strict_summary.nc"),
        ],
        "reference": [p_rel("projects/era5_land/notebooks/figures_era5l_bars/bars_snow_combined_3x3_era5l_only.png")],
        "upstream": "projects/era5_land/notebooks/plot_ERA5L_comparison_bars.ipynb",
    },
]

pd.DataFrame([{k: v for k, v in row.items() if k not in {"required", "reference"}} for row in figure_registry])

## Availability Checks

These checks intentionally run before any plotting. A missing file does not necessarily block every figure, but it should be resolved or explicitly accepted before rebuilding that figure.

In [ ]:
def file_status(path: Path) -> dict:
    p = Path(path)
    exists = p.exists()
    return {
        "path": str(p),
        "exists": exists,
        "size_mb": round(p.stat().st_size / 1024**2, 3) if exists and p.is_file() else np.nan,
        "mtime": pd.to_datetime(p.stat().st_mtime, unit="s") if exists else pd.NaT,
    }


availability_rows = []
for fig in figure_registry:
    for role in ("required", "reference"):
        paths = fig.get(role, [])
        if not paths:
            continue
        for path in paths:
            row = file_status(path)
            row.update({
                "figure": fig["figure"],
                "role": role,
                "description": fig["description"],
                "upstream": fig["upstream"],
            })
            availability_rows.append(row)

availability = pd.DataFrame(availability_rows)[[
    "figure", "role", "exists", "size_mb", "mtime", "description", "path", "upstream"
]]

missing_required = availability[(availability["role"] == "required") & (~availability["exists"])]

display(availability.sort_values(["figure", "role", "path"]))
if len(missing_required):
    print("Missing required products:")
    display(missing_required[["figure", "description", "path", "upstream"]])
else:
    print("All registered required products are present.")

## Fine-Period OFA Temporal-Stats Availability

Fig. 6 currently depends on precomputed period-split `temporal_stats_*` files. The Aqua transition can be shown from monthly count pickles, but separate P1/P2 OmF maps require matching period-split temporal stats if we want rows for those exact periods.

In [ ]:
def ymd(ts: pd.Timestamp) -> str:
    return ts.strftime("%Y%m%d")


period_stat_rows = []
for _, row in fine_periods.iterrows():
    for exp in ("OL", "DA"):
        path = p_diag(f"temporal_stats_{exp}_{ymd(row['start'])}_{ymd(row['end'])}.nc4")
        status = file_status(path)
        status.update({
            "period_id": row["period_id"],
            "period_label": row["label"],
            "exp": exp,
        })
        period_stat_rows.append(status)

period_stats_availability = pd.DataFrame(period_stat_rows)[[
    "period_id", "period_label", "exp", "exists", "size_mb", "mtime", "path"
]]
display(period_stats_availability)

missing_period_stats = period_stats_availability[~period_stats_availability["exists"]]
if len(missing_period_stats):
    print("Fine-period temporal_stats files missing. Fig. 6 can use existing coarse splits or these can be regenerated.")
    display(missing_period_stats[["period_id", "period_label", "exp", "path"]])
else:
    print("All fine-period temporal_stats files are present.")

## Figure 1: Observing-System Timeline

This section writes the shared observing-system timeline used to label the fine-grain periods in later OFA figures.


In [ ]:
import csv
import textwrap

import matplotlib.dates as mdates
import matplotlib.pyplot as plt
from matplotlib.patches import Patch

FIGURE_MANIFEST = []


def record_figure(fig_id: str, path: Path, sources: list[str], settings: dict | None = None) -> None:
    FIGURE_MANIFEST.append({
        "figure": fig_id,
        "path": str(path),
        "exists": path.exists(),
        "size_mb": round(path.stat().st_size / 1024**2, 3) if path.exists() else np.nan,
        "sources": "; ".join(sources),
        "settings": settings or {},
    })


def save_manifest(path: Path = PAPER_FIG_DIR / "paper_figures_manifest.csv") -> pd.DataFrame:
    manifest = pd.DataFrame(FIGURE_MANIFEST)
    if len(manifest):
        manifest["settings"] = manifest["settings"].map(lambda value: repr(value))
        manifest.to_csv(path, index=False)
    return manifest


def panel_label(ax, label: str, x: float = -0.105, y: float = 1.045) -> None:
    ax.text(
        x,
        y,
        label,
        transform=ax.transAxes,
        ha="left",
        va="bottom",
        fontsize=11,
        fontweight="bold",
        clip_on=False,
    )


period_colors = {
    "P1": "#f2efe6",
    "P2": "#e7f0f7",
    "P3": "#eef4e8",
    "P4": "#f7eadf",
    "P5": "#ebe7f3",
    "P6": "#e8f3f1",
    "P7": "#f5edf3",
    "P8": "#edf0f5",
    "P9": "#f2f0e8",
}

fig, ax = plt.subplots(figsize=(11.2, 5.6), constrained_layout=True)

# Period shading and compact labels. The full period names live in the registry;
# this figure uses short IDs to avoid crowding the narrow late-period bands.
for _, period in fine_periods.iterrows():
    ax.axvspan(period["start"], period["end"] + pd.Timedelta(days=1),
               color=period_colors[period["period_id"]], zorder=0)
    center = period["start"] + (period["end"] - period["start"]) / 2
    ax.text(center, 8.55, period["period_id"], ha="center", va="bottom", fontsize=8.5, weight="bold")

# Sensor availability bars.
y_positions = np.arange(len(sensor_rows))[::-1]
for y, (_, row) in zip(y_positions, sensor_rows.iterrows()):
    ax.barh(y, row["end"] - row["start"], left=row["start"], height=0.58,
            color=row["color"], edgecolor="0.25", linewidth=0.7, zorder=2)
    ax.text(row["start"] + pd.Timedelta(days=35), y, row["sensor"], va="center", ha="left",
            fontsize=9, color="white" if row["group"] != "Snow cover" else "#102030", weight="bold")

# Period boundaries.
for _, period in fine_periods.iloc[1:].iterrows():
    ax.axvline(period["start"], color="0.35", linewidth=0.8, linestyle="--", zorder=1)

ax.set_xlim(PAPER_START, PAPER_END + pd.Timedelta(days=45))
ax.set_ylim(-0.8, 9.25)
ax.set_yticks([])
ax.xaxis.set_major_locator(mdates.YearLocator(2))
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
ax.tick_params(axis="x", labelsize=9)
ax.set_xlabel("Year")
ax.set_title("M21C Land Sweeper assimilated observing-system timeline", loc="left", fontsize=13, pad=16)
ax.spines[["left", "right", "top"]].set_visible(False)
ax.grid(axis="x", color="white", linewidth=0.8, zorder=0)

legend_handles = [Patch(facecolor=color, edgecolor="0.25", label=group) for group, color in [
    ("Snow cover", "#4c78a8"),
    ("Active microwave", "#f58518"),
    ("Passive microwave", "#54a24b"),
    ("GNSS-R", "#ff9da6"),
]]
ax.legend(handles=legend_handles, loc="lower center", bbox_to_anchor=(0.5, -0.22), ncol=4,
          frameon=False, fontsize=8.5)

fig1_png = PAPER_FIG_DIR / "fig01_observing_system_timeline.png"
fig1_pdf = PAPER_FIG_DIR / "fig01_observing_system_timeline.pdf"
fig.savefig(fig1_png, dpi=300, bbox_inches="tight")
fig.savefig(fig1_pdf, bbox_inches="tight")
plt.show()

record_figure(
    "Fig. 1",
    fig1_png,
    sources=["projects/M21C_ls/config/observing_system_registry.json"],
    settings={"paper_start": str(PAPER_START.date()), "paper_end": str(PAPER_END.date()), "format": "png", "dpi": 300},
)
record_figure(
    "Fig. 1",
    fig1_pdf,
    sources=["projects/M21C_ls/config/observing_system_registry.json"],
    settings={"paper_start": str(PAPER_START.date()), "paper_end": str(PAPER_END.date()), "format": "pdf"},
)
manifest = save_manifest()
display(manifest)


## Shared OFA Helpers

These helpers load the full-period ObsFcstAna products and convert tile-level values onto the M36 EASE grid for the OFA map figures.


In [ ]:
import pickle

from netCDF4 import Dataset

sys.path.append(str(REPO_ROOT / "common/python/io"))
from read_GEOSldas import read_tilecoord

import cartopy.crs as ccrs
import cartopy.feature as cfeature

EASE_PATH = REPO_ROOT / "common/python/plotting/ease_grids"

SPECIES_GROUPS = {
    "SMOS": [0, 1, 2, 3],
    "SMAP": [4, 5, 6, 7],
    "ASCAT": [8, 9, 10],
    "CYGN": [13],
    "MODIS": [11, 12],
}
PANEL_GROUPS = ["MODIS", "ASCAT", "SMOS", "SMAP", "CYGN"]
DISPLAY_NAME = {"CYGN": "CYGNSS"}
PANEL_LABELS = ["a", "b", "c", "d", "e"]
GROUP_COLORS = {
    "MODIS": "#4c78a8",
    "ASCAT": "#f58518",
    "SMOS": "#54a24b",
    "SMAP": "#9d755d",
    "CYGN": "#ff9da6",
}
NMIN = 20


def load_temporal_stats(path: Path) -> dict[str, np.ndarray]:
    stats = {}
    with Dataset(path, "r") as nc:
        for key, value in nc.variables.items():
            arr = value[:]
            if hasattr(arr, "filled"):
                arr = arr.filled(np.nan)
            stats[key] = np.asarray(arr, dtype=float)
    return stats


def weighted_group_metrics_from_temporal(stats: dict[str, np.ndarray], nmin: int = NMIN) -> dict[str, dict[str, np.ndarray]]:
    n_data = stats["N_data"].copy()
    metrics = {key: stats[key].copy() for key in ["OmF_mean", "OmF_stdv", "OmF_norm_mean", "OmF_norm_stdv", "OmA_mean", "OmA_stdv"]}
    for key in metrics:
        metrics[key][n_data < nmin] = np.nan
    n_data[n_data < nmin] = 0

    grouped = {}
    for group, species_indices in SPECIES_GROUPS.items():
        weights = n_data[:, species_indices]
        group_n = np.nansum(weights, axis=1)
        grouped[group] = {"Nobs_data": group_n}
        for key, arr in metrics.items():
            numerator = np.nansum(arr[:, species_indices] * weights, axis=1)
            grouped[group][key] = np.divide(
                numerator,
                group_n,
                out=np.full(group_n.shape, np.nan, dtype=float),
                where=group_n > 0,
            )
    return grouped


def load_monthly_stats(path: Path) -> tuple[dict[str, np.ndarray], list[pd.Timestamp]]:
    with open(path, "rb") as f:
        stats = pickle.load(f)
    dates = [pd.to_datetime(value, format="%Y%m") for value in stats["date_vec"]]
    arrays = {key: np.asarray(value, dtype=float) for key, value in stats.items() if key != "date_vec"}
    return arrays, dates


def weighted_group_metrics_from_monthly(stats: dict[str, np.ndarray]) -> dict[str, dict[str, np.ndarray]]:
    grouped = {}
    for group, species_indices in SPECIES_GROUPS.items():
        weights = stats["N_data"][:, species_indices]
        group_n = np.nansum(weights, axis=1)
        grouped[group] = {"N_data": group_n}
        for key in ["O_mean", "F_mean", "OmF_mean", "OmF_stdv", "OmA_mean", "OmA_stdv"]:
            numerator = np.nansum(stats[key][:, species_indices] * weights, axis=1)
            grouped[group][key] = np.divide(
                numerator,
                group_n,
                out=np.full(group_n.shape, np.nan, dtype=float),
                where=group_n > 0,
            )
    return grouped


stats_ol_full = load_temporal_stats(p_diag("temporal_stats_OL_20000601_20240531.nc4"))
stats_da_full = load_temporal_stats(p_diag("temporal_stats_DA_20000601_20240531.nc4"))
group_metrics_ol_full = weighted_group_metrics_from_temporal(stats_ol_full)
group_metrics_da_full = weighted_group_metrics_from_temporal(stats_da_full)

monthly_ol, date_vec_ol = load_monthly_stats(p_diag("spatial_stats_OL_200006_202405.pkl"))
monthly_da, date_vec_da = load_monthly_stats(p_diag("spatial_stats_DA_200006_202405.pkl"))
group_monthly_ol = weighted_group_metrics_from_monthly(monthly_ol)
group_monthly_da = weighted_group_metrics_from_monthly(monthly_da)

# M36 EASE grid and tile index map.
tilecoord = read_tilecoord(str(p_diag("LS_OLv8_M36.ldas_tilecoord.bin")))
lats2d = np.fromfile(str(EASE_PATH / "EASE2_M36km.lats.964x406x1.double"), dtype=np.float64).reshape((406, 964))
lons2d = np.fromfile(str(EASE_PATH / "EASE2_M36km.lons.964x406x1.double"), dtype=np.float64).reshape((406, 964))
tile_rows = np.asarray(tilecoord["j_indg"], dtype=int)
tile_cols = np.asarray(tilecoord["i_indg"], dtype=int)


def grid_from_tile_values(values: np.ndarray, mask_antarctica: bool = True) -> np.ndarray:
    grid = np.full(lats2d.shape, np.nan, dtype=float)
    vals = np.asarray(values, dtype=float)
    valid = np.isfinite(vals)
    grid[tile_rows[valid], tile_cols[valid]] = vals[valid]
    if mask_antarctica:
        grid = np.where(lats2d < -60.0, np.nan, grid)
    return grid


def add_map_base(ax) -> None:
    ax.add_feature(cfeature.LAND, facecolor="0.92", edgecolor="none", zorder=0)
    ax.coastlines(resolution="50m", linewidth=0.45)
    ax.set_global()
    ax.set_extent([-180, 180, -60, 90], crs=ccrs.PlateCarree())


def shade_fine_periods(ax, label_periods: bool = False, y_text: float = 0.96) -> None:
    for _, period in fine_periods.iterrows():
        ax.axvspan(period["start"], period["end"] + pd.Timedelta(days=1),
                   color=period_colors[period["period_id"]], zorder=0)
        if label_periods:
            center = period["start"] + (period["end"] - period["start"]) / 2
            ax.text(center, y_text, period["period_id"], ha="center", va="top",
                    transform=ax.get_xaxis_transform(), fontsize=8, weight="bold")
    for _, period in fine_periods.iloc[1:].iterrows():
        ax.axvline(period["start"], color="0.55", linewidth=0.7, linestyle="--", zorder=1)

print("Loaded OFA temporal/monthly stats and M36 tile grid.")


## Figure 2: Mean Assimilated Observations Per Day

Global maps of mean DA assimilated-observation counts per active observing-system day. Counts are tile-space monthly/full-period ObsFcstAna counts grouped by observation species.


In [ ]:
# Active-day divisors inherited from the legacy plotting notebook. These reflect data availability/QC windows, not simply mission launch-to-end dates.
obs_days_by_group = {
    "MODIS": 8700,
    "ASCAT": 6121,
    "SMOS": 5047,
    "SMAP": 3287,
    "CYGN": 2131,
}

fig, axs = plt.subplots(
    3,
    2,
    figsize=(13.5, 11.5),
    subplot_kw={"projection": ccrs.Robinson()},
    constrained_layout=True,
)
axs_flat = axs.ravel()
mesh = None
summary_rows = []

for i, group in enumerate(PANEL_GROUPS):
    ax = axs_flat[i]
    n_days = obs_days_by_group[group]
    obs_per_day = group_metrics_da_full[group]["Nobs_data"] / float(n_days)
    grid = grid_from_tile_values(obs_per_day)
    finite = np.isfinite(grid)

    mesh = ax.pcolormesh(
        lons2d,
        lats2d,
        grid,
        transform=ccrs.PlateCarree(),
        cmap="viridis",
        vmin=0,
        vmax=2,
        shading="auto",
        rasterized=True,
    )
    add_map_base(ax)
    name = DISPLAY_NAME.get(group, group)
    ax.set_title(name, fontsize=10.5, loc="left")
    panel_label(ax, f"({PANEL_LABELS[i]})")
    mean_val = float(np.nanmean(grid)) if finite.any() else np.nan
    std_val = float(np.nanstd(grid)) if finite.any() else np.nan
    ax.text(
        0.02,
        0.03,
        f"mean {mean_val:.2f} +/- {std_val:.2f} d$^{{-1}}$",
        transform=ax.transAxes,
        fontsize=8.5,
        bbox={"boxstyle": "round,pad=0.2", "facecolor": "white", "alpha": 0.82, "edgecolor": "none"},
    )
    summary_rows.append({"group": group, "active_days": n_days, "spatial_mean_obs_per_day": mean_val, "spatial_std_obs_per_day": std_val})

axs_flat[-1].axis("off")
cbar = fig.colorbar(mesh, ax=axs_flat.tolist(), orientation="horizontal", fraction=0.045, pad=0.035)
cbar.set_label("Assimilated observations per day")
cbar.set_ticks(np.linspace(0, 2, 5))
fig.suptitle("Mean assimilated observations per day by observing system", fontsize=13)

fig2_png = PAPER_FIG_DIR / "fig02_mean_assimilated_observations_per_day.png"
fig2_pdf = PAPER_FIG_DIR / "fig02_mean_assimilated_observations_per_day.pdf"
fig.savefig(fig2_png, dpi=300, bbox_inches="tight")
fig.savefig(fig2_pdf, bbox_inches="tight")
plt.show()

fig2_summary = pd.DataFrame(summary_rows)
fig2_summary.to_csv(PAPER_FIG_DIR / "fig02_obs_per_day_summary.csv", index=False)
display(fig2_summary)

record_figure("Fig. 2", fig2_png, sources=["temporal_stats_DA_20000601_20240531.nc4", "LS_OLv8_M36.ldas_tilecoord.bin"], settings={"metric": "DA N_data / active days", "format": "png", "dpi": 300})
record_figure("Fig. 2", fig2_pdf, sources=["temporal_stats_DA_20000601_20240531.nc4", "LS_OLv8_M36.ldas_tilecoord.bin"], settings={"metric": "DA N_data / active days", "format": "pdf"})
manifest = save_manifest()


## Figure 3: Monthly Assimilated Observation Counts

Monthly DA observation-count time series with the same P1-P9 shading used in Fig. 1.


In [ ]:
for group in PANEL_GROUPS:
    counts = group_monthly_da[group]["N_data"].copy()
    group_monthly_da[group]["N_data_plot"] = np.where(counts == 0, np.nan, counts)

group_stack = np.vstack([group_monthly_da[group]["N_data_plot"] for group in PANEL_GROUPS])
total_obs = np.nansum(group_stack, axis=0)

fig, (ax_top, ax_bottom) = plt.subplots(
    2,
    1,
    figsize=(11.5, 7.2),
    sharex=True,
    height_ratios=[1.05, 1.15],
    constrained_layout=True,
)

for ax in (ax_top, ax_bottom):
    shade_fine_periods(ax, label_periods=ax is ax_top)

ax_top.plot(date_vec_da, total_obs, color="0.1", lw=1.4)
ax_top.set_ylabel("Total obs month$^{-1}$")
ax_top.set_title("Monthly assimilated observation counts", loc="left", fontsize=12.5)
panel_label(ax_top, "(a)")

for group in PANEL_GROUPS:
    ax_bottom.plot(
        date_vec_da,
        group_monthly_da[group]["N_data_plot"],
        label=DISPLAY_NAME.get(group, group),
        lw=1.2,
        color=GROUP_COLORS[group],
    )
ax_bottom.set_ylabel("Obs month$^{-1}$")
ax_bottom.set_xlabel("Year")
panel_label(ax_bottom, "(b)")
ax_bottom.legend(ncol=5, fontsize=8.5, loc="upper center", bbox_to_anchor=(0.5, -0.18), frameon=False)
ax_bottom.set_xlim(PAPER_START, PAPER_END)
ax_bottom.xaxis.set_major_locator(mdates.YearLocator(2))
ax_bottom.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))

fig3_png = PAPER_FIG_DIR / "fig03_monthly_assimilated_observation_counts.png"
fig3_pdf = PAPER_FIG_DIR / "fig03_monthly_assimilated_observation_counts.pdf"
fig.savefig(fig3_png, dpi=300, bbox_inches="tight")
fig.savefig(fig3_pdf, bbox_inches="tight")
plt.show()

record_figure("Fig. 3", fig3_png, sources=["spatial_stats_DA_200006_202405.pkl", "projects/M21C_ls/config/observing_system_registry.json"], settings={"metric": "monthly DA N_data", "format": "png", "dpi": 300})
record_figure("Fig. 3", fig3_pdf, sources=["spatial_stats_DA_200006_202405.pkl", "projects/M21C_ls/config/observing_system_registry.json"], settings={"metric": "monthly DA N_data", "format": "pdf"})
manifest = save_manifest()


## Figure 4: Full-Period O-F StdDev Improvement Maps

Maps of full-period normalized O-F StdDev change by observing system. Positive values indicate lower DA O-F StdDev than OL.


In [ ]:
fig, axs = plt.subplots(
    3,
    2,
    figsize=(13.5, 11.5),
    subplot_kw={"projection": ccrs.Robinson()},
    constrained_layout=True,
)
axs_flat = axs.ravel()
mesh = None
summary_rows = []

for i, group in enumerate(PANEL_GROUPS):
    ax = axs_flat[i]
    ol = group_metrics_ol_full[group]["OmF_stdv"]
    da = group_metrics_da_full[group]["OmF_stdv"]
    valid = np.isfinite(ol) & np.isfinite(da) & (ol > 1e-6)
    improvement = np.divide(
        ol - da,
        ol,
        out=np.full_like(ol, np.nan, dtype=float),
        where=valid,
    ) * 100.0
    grid = grid_from_tile_values(improvement)
    finite = np.isfinite(grid)

    mesh = ax.pcolormesh(
        lons2d,
        lats2d,
        grid,
        transform=ccrs.PlateCarree(),
        cmap="RdBu_r",
        vmin=-60,
        vmax=60,
        shading="auto",
        rasterized=True,
    )
    add_map_base(ax)
    name = DISPLAY_NAME.get(group, group)
    ax.set_title(name, fontsize=10.5, loc="left")
    panel_label(ax, f"({PANEL_LABELS[i]})")
    mean_val = float(np.nanmean(grid)) if finite.any() else np.nan
    std_val = float(np.nanstd(grid)) if finite.any() else np.nan
    ax.text(
        0.02,
        0.03,
        f"mean {mean_val:.1f} +/- {std_val:.1f}%",
        transform=ax.transAxes,
        fontsize=8.5,
        bbox={"boxstyle": "round,pad=0.2", "facecolor": "white", "alpha": 0.82, "edgecolor": "none"},
    )
    summary_rows.append({"group": group, "spatial_mean_percent": mean_val, "spatial_std_percent": std_val})

axs_flat[-1].axis("off")
cbar = fig.colorbar(mesh, ax=axs_flat.tolist(), orientation="horizontal", fraction=0.045, pad=0.035)
cbar.set_label("O-F StdDev improvement, (OL - DA) / OL (%)")
cbar.set_ticks(np.arange(-60, 61, 20))
fig.suptitle("Full-period DA improvement in O-F standard deviation", fontsize=13)

fig4_png = PAPER_FIG_DIR / "fig04_full_period_omf_stddev_improvement.png"
fig4_pdf = PAPER_FIG_DIR / "fig04_full_period_omf_stddev_improvement.pdf"
fig.savefig(fig4_png, dpi=300, bbox_inches="tight")
fig.savefig(fig4_pdf, bbox_inches="tight")
plt.show()

fig4_summary = pd.DataFrame(summary_rows)
fig4_summary.to_csv(PAPER_FIG_DIR / "fig04_omf_stddev_improvement_summary.csv", index=False)
display(fig4_summary)

record_figure("Fig. 4", fig4_png, sources=["temporal_stats_OL_20000601_20240531.nc4", "temporal_stats_DA_20000601_20240531.nc4", "LS_OLv8_M36.ldas_tilecoord.bin"], settings={"metric": "(OL - DA) / OL OmF_stdv", "valid_mask": "OL OmF_stdv > 1e-6", "format": "png", "dpi": 300, "color_convention": "red = improvement"})
record_figure("Fig. 4", fig4_pdf, sources=["temporal_stats_OL_20000601_20240531.nc4", "temporal_stats_DA_20000601_20240531.nc4", "LS_OLv8_M36.ldas_tilecoord.bin"], settings={"metric": "(OL - DA) / OL OmF_stdv", "valid_mask": "OL OmF_stdv > 1e-6", "format": "pdf", "color_convention": "red = improvement"})
manifest = save_manifest()


## Figure 5: Monthly O-F StdDev Improvement Evolution

Monthly observing-system time series of O-F StdDev improvement, using the same positive-is-improvement convention as Fig. 4.

In [ ]:
fig5_groups = ["MODIS", "ASCAT", "SMOS", "SMAP", "CYGN"]
fig5_rows = []
period_rows = []

for group in fig5_groups:
    ol = group_monthly_ol[group]["OmF_stdv"]
    da = group_monthly_da[group]["OmF_stdv"]
    valid = (
        np.isfinite(ol)
        & np.isfinite(da)
        & (ol > 1e-6)
        & (group_monthly_ol[group]["N_data"] > 0)
        & (group_monthly_da[group]["N_data"] > 0)
    )
    improvement = np.divide(
        ol - da,
        ol,
        out=np.full_like(ol, np.nan, dtype=float),
        where=valid,
    ) * 100.0
    series = pd.Series(improvement, index=pd.DatetimeIndex(date_vec_da), name=group)
    smooth = series.rolling(5, center=True, min_periods=2).mean()
    for date, value in series.items():
        fig5_rows.append({"date": date, "group": group, "improvement_percent": value})
    for _, period in fine_periods.iterrows():
        period_values = series.loc[(series.index >= period["start"]) & (series.index <= period["end"])]
        period_rows.append({
            "period_id": period["period_id"],
            "period_label": period["label"],
            "group": group,
            "mean_improvement_percent": float(period_values.mean(skipna=True)),
            "n_months": int(period_values.notna().sum()),
        })

fig5_monthly = pd.DataFrame(fig5_rows)
fig5_period_summary = pd.DataFrame(period_rows)
fig5_monthly.to_csv(PAPER_FIG_DIR / "fig05_monthly_omf_stddev_improvement_timeseries.csv", index=False)
fig5_period_summary.to_csv(PAPER_FIG_DIR / "fig05_period_mean_omf_stddev_improvement.csv", index=False)

fig, (ax, ax_period) = plt.subplots(
    2,
    1,
    figsize=(11.5, 7.2),
    height_ratios=[3.5, 1.15],
    constrained_layout=True,
)

shade_fine_periods(ax, label_periods=True, y_text=0.97)
for group in fig5_groups:
    name = DISPLAY_NAME.get(group, group)
    series = fig5_monthly.loc[fig5_monthly["group"] == group].set_index("date")["improvement_percent"]
    smooth = series.rolling(5, center=True, min_periods=2).mean()
    color = GROUP_COLORS[group]
    ax.plot(series.index, series.values, color=color, linewidth=0.75, alpha=0.28, zorder=2)
    ax.plot(smooth.index, smooth.values, color=color, linewidth=2.0, label=name, zorder=3)

ax.axhline(0, color="0.15", linewidth=0.9, linestyle=":", zorder=2)
ax.set_xlim(PAPER_START, PAPER_END)
ax.set_ylim(-5, 32)
ax.set_xlabel("Year")
ax.set_ylabel("O-F StdDev improvement (%)")
ax.set_title("Monthly DA improvement in O-F standard deviation", fontsize=13)
panel_label(ax, "(a)")
ax.text(
    0.01,
    0.04,
    "positive = lower DA O-F StdDev than OL",
    transform=ax.transAxes,
    fontsize=8.5,
    bbox={"boxstyle": "round,pad=0.25", "facecolor": "white", "alpha": 0.86, "edgecolor": "none"},
)
ax.legend(ncols=5, loc="upper center", bbox_to_anchor=(0.5, -0.08), frameon=False)
ax.grid(axis="y", alpha=0.25, linewidth=0.7)

period_matrix = fig5_period_summary.pivot(index="group", columns="period_id", values="mean_improvement_percent").reindex(fig5_groups)
masked_matrix = np.ma.masked_invalid(period_matrix.values)
im = ax_period.imshow(masked_matrix, aspect="auto", cmap="RdBu_r", vmin=-25, vmax=25)
ax_period.set_yticks(np.arange(len(fig5_groups)))
ax_period.set_yticklabels([DISPLAY_NAME.get(group, group) for group in fig5_groups])
ax_period.set_xticks(np.arange(len(fine_periods)))
ax_period.set_xticklabels(fine_periods["period_id"])
ax_period.set_ylabel("Period mean")
panel_label(ax_period, "(b)")
for row_idx, group in enumerate(fig5_groups):
    for col_idx, period_id in enumerate(fine_periods["period_id"]):
        value = period_matrix.loc[group, period_id]
        if np.isfinite(value):
            ax_period.text(col_idx, row_idx, f"{value:.1f}", ha="center", va="center", fontsize=7.5)
        else:
            ax_period.text(col_idx, row_idx, "--", ha="center", va="center", fontsize=7.5, color="0.35")
for spine in ax_period.spines.values():
    spine.set_visible(False)
ax_period.tick_params(axis="both", length=0)
cbar = fig.colorbar(im, ax=ax_period, orientation="horizontal", fraction=0.22, pad=0.18)
cbar.set_label("Period-mean improvement (%)")

ax_period.xaxis.set_ticks_position("top")
ax_period.xaxis.set_label_position("top")
ax.xaxis.set_major_locator(mdates.YearLocator(base=2))
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
ax.tick_params(axis="x", rotation=0)

fig5_png = PAPER_FIG_DIR / "fig05_monthly_omf_stddev_improvement_evolution.png"
fig5_pdf = PAPER_FIG_DIR / "fig05_monthly_omf_stddev_improvement_evolution.pdf"
fig.savefig(fig5_png, dpi=300, bbox_inches="tight")
fig.savefig(fig5_pdf, bbox_inches="tight")
plt.show()

display(fig5_period_summary)
record_figure(
    "Fig. 5",
    fig5_png,
    sources=["spatial_stats_OL_200006_202405.pkl", "spatial_stats_DA_200006_202405.pkl"],
    settings={
        "metric": "(OL - DA) / OL monthly OmF_stdv",
        "valid_mask": "OL OmF_stdv > 1e-6 and monthly OL/DA N_data > 0",
        "smoothing": "5-month centered rolling mean over monthly values",
        "groups": fig5_groups,
        "periods": "fine_periods P1-P9",
        "color_convention": "positive/red = improvement",
        "format": "png",
        "dpi": 300,
    },
)
record_figure(
    "Fig. 5",
    fig5_pdf,
    sources=["spatial_stats_OL_200006_202405.pkl", "spatial_stats_DA_200006_202405.pkl"],
    settings={
        "metric": "(OL - DA) / OL monthly OmF_stdv",
        "valid_mask": "OL OmF_stdv > 1e-6 and monthly OL/DA N_data > 0",
        "smoothing": "5-month centered rolling mean over monthly values",
        "groups": fig5_groups,
        "periods": "fine_periods P1-P9",
        "color_convention": "positive/red = improvement",
        "format": "pdf",
    },
)
manifest = save_manifest()


## Figure 6: Period-by-Sensor O-F StdDev Improvement Maps

Availability-aware P1-P9 map matrix of O-F StdDev improvement by observing system. Positive values use the same convention as Figs. 4-5: lower DA O-F StdDev than OL.

In [ ]:
fig6_groups = ["MODIS", "ASCAT", "SMOS", "SMAP", "CYGN"]
fig6_valid_omf_std_min = 1e-3
fig6_vmin, fig6_vmax = -60, 60
fig6_summary_rows = []
fig6_sources = []

fig, axs = plt.subplots(
    len(fine_periods),
    len(fig6_groups),
    figsize=(14.2, 15.4),
    subplot_kw={"projection": ccrs.Robinson()},
)
axs = np.atleast_2d(axs)
mesh = None

for row_idx, (_, period) in enumerate(fine_periods.iterrows()):
    start = ymd(period["start"])
    end = ymd(period["end"])
    ol_path = p_diag(f"temporal_stats_OL_{start}_{end}.nc4")
    da_path = p_diag(f"temporal_stats_DA_{start}_{end}.nc4")
    fig6_sources.extend([ol_path.name, da_path.name])

    ol_metrics = weighted_group_metrics_from_temporal(load_temporal_stats(ol_path))
    da_metrics = weighted_group_metrics_from_temporal(load_temporal_stats(da_path))

    for col_idx, group in enumerate(fig6_groups):
        ax = axs[row_idx, col_idx]
        display_name = DISPLAY_NAME.get(group, group)
        ol = ol_metrics[group]["OmF_stdv"]
        da = da_metrics[group]["OmF_stdv"]
        ol_n = ol_metrics[group]["Nobs_data"]
        da_n = da_metrics[group]["Nobs_data"]
        valid = (
            np.isfinite(ol)
            & np.isfinite(da)
            & (ol > fig6_valid_omf_std_min)
            & (ol_n > 0)
            & (da_n > 0)
        )
        improvement = np.divide(
            ol - da,
            ol,
            out=np.full_like(ol, np.nan, dtype=float),
            where=valid,
        ) * 100.0
        valid_values = improvement[valid]

        if valid_values.size:
            grid = grid_from_tile_values(improvement)
            mesh = ax.pcolormesh(
                lons2d,
                lats2d,
                grid,
                transform=ccrs.PlateCarree(),
                cmap="RdBu_r",
                vmin=fig6_vmin,
                vmax=fig6_vmax,
                shading="auto",
                rasterized=True,
            )
            add_map_base(ax)
            mean_val = float(np.nanmean(valid_values))
            std_val = float(np.nanstd(valid_values))
            n_tiles = int(valid_values.size)
            ax.text(
                0.02,
                0.04,
                f"{mean_val:.1f} +/- {std_val:.1f}%",
                transform=ax.transAxes,
                fontsize=6.5,
                bbox={"boxstyle": "round,pad=0.16", "facecolor": "white", "alpha": 0.84, "edgecolor": "none"},
            )
            available = True
        else:
            ax.set_axis_off()
            mean_val = np.nan
            std_val = np.nan
            n_tiles = 0
            available = False
            ax.text(
                0.5,
                0.52,
                "not\nassimilated",
                transform=ax.transAxes,
                ha="center",
                va="center",
                fontsize=7.0,
                color="0.48",
                linespacing=1.1,
            )

        if row_idx == 0:
            ax.set_title(display_name, fontsize=10.5, pad=3)
        if col_idx == 0:
            ax.text(
                -0.12,
                0.5,
                f"{period['period_id']}\n{period['start']:%Y-%m}\nto\n{period['end']:%Y-%m}",
                transform=ax.transAxes,
                ha="right",
                va="center",
                fontsize=7.0,
                weight="bold",
                linespacing=1.05,
            )

        fig6_summary_rows.append({
            "period_id": period["period_id"],
            "period_label": period["label"],
            "start": period["start"].date().isoformat(),
            "end": period["end"].date().isoformat(),
            "group": group,
            "available": available,
            "n_tiles": n_tiles,
            "spatial_mean_percent": mean_val,
            "spatial_std_percent": std_val,
            "valid_mask": f"OL OmF_stdv > {fig6_valid_omf_std_min:g}; OL/DA group Nobs_data > 0",
        })

if mesh is None:
    raise RuntimeError("No available Fig. 6 panels were plotted.")

fig.suptitle("DA improvement in O-F standard deviation by observing-system period", fontsize=13, y=0.985)
plt.subplots_adjust(left=0.075, right=0.995, top=0.955, bottom=0.07, wspace=0.01, hspace=0.02)
cbar = fig.colorbar(mesh, ax=axs.ravel().tolist(), orientation="horizontal", fraction=0.024, pad=0.018)
cbar.set_label("O-F StdDev improvement, (OL - DA) / OL (%)")
cbar.set_ticks(np.arange(fig6_vmin, fig6_vmax + 1, 20))

fig6_png = PAPER_FIG_DIR / "fig06_period_sensor_omf_stddev_improvement_maps.png"
fig6_pdf = PAPER_FIG_DIR / "fig06_period_sensor_omf_stddev_improvement_maps.pdf"
fig.savefig(fig6_png, dpi=300, bbox_inches="tight")
fig.savefig(fig6_pdf, bbox_inches="tight")
plt.show()

fig6_summary = pd.DataFrame(fig6_summary_rows)
fig6_summary.to_csv(PAPER_FIG_DIR / "fig06_period_sensor_omf_stddev_improvement_summary.csv", index=False)
display(fig6_summary)

fig6_unique_sources = sorted(set(fig6_sources + ["LS_OLv8_M36.ldas_tilecoord.bin"]))
record_figure(
    "Fig. 6",
    fig6_png,
    sources=fig6_unique_sources,
    settings={
        "metric": "(OL - DA) / OL period OmF_stdv",
        "valid_mask": f"OL OmF_stdv > {fig6_valid_omf_std_min:g} and period OL/DA Nobs_data > 0",
        "groups": fig6_groups,
        "periods": "fine_periods P1-P9",
        "color_convention": "positive/red = improvement",
        "format": "png",
        "dpi": 300,
    },
)
record_figure(
    "Fig. 6",
    fig6_pdf,
    sources=fig6_unique_sources,
    settings={
        "metric": "(OL - DA) / OL period OmF_stdv",
        "valid_mask": f"OL OmF_stdv > {fig6_valid_omf_std_min:g} and period OL/DA Nobs_data > 0",
        "groups": fig6_groups,
        "periods": "fine_periods P1-P9",
        "color_convention": "positive/red = improvement",
        "format": "pdf",
    },
)
manifest = save_manifest()


## Figure 7: ISMN Soil-Moisture Skill by Validation Period

Paired DA-minus-OL skill changes from the precomputed ISMN hybrid network table. The broader validation periods map onto the P1-P9 observing-system registry.

In [ ]:
fig7_table_path = p_rel("projects/M21C_ls/output/ismn_network_skill/batch_figures/all_networks_hybrid_OL_DA_delta_surface_rz_R_anomR_ubRMSE_table.csv")
fig7_df = pd.read_csv(fig7_table_path)

fig7_validation_defs = {
    "pre-ASCAT": {"id": "V1", "name": "SCF-only period", "fine_periods": "P1–P2", "axis_label": "SCF-only period\nP1–P2"},
    "pre-SMAP": {"id": "V2", "name": "Pre-SMAP microwave", "fine_periods": "P3–P5", "axis_label": "Pre-SMAP microwave\nP3–P5"},
    "SMAP-era": {"id": "V3", "name": "SMAP-era microwave", "fine_periods": "P6–P9", "axis_label": "SMAP-era microwave\nP6–P9"},
}
fig7_window_order = ["pre-ASCAT", "pre-SMAP", "SMAP-era"]
fig7_metric_order = ["R", "anomR", "ubRMSE"]
fig7_domain_order = ["surface", "rz"]
fig7_domain_labels = {"surface": "Surface", "rz": "Root zone"}
fig7_metric_labels = {
    "R": {"title": "R", "ylabel": "Delta R"},
    "anomR": {"title": "Anomaly R", "ylabel": "Delta anomaly R"},
    "ubRMSE": {"title": "ubRMSE", "ylabel": r"Delta ubRMSE ($m^3$ $m^{-3}$)"},
}
fig7_base_ylim = {
    ("surface", "R"): (-0.04, 0.18),
    ("surface", "anomR"): (-0.05, 0.22),
    ("surface", "ubRMSE"): (-0.004, 0.014),
    ("rz", "R"): (-0.05, 0.07),
    ("rz", "anomR"): (-0.08, 0.09),
    ("rz", "ubRMSE"): (-0.008, 0.006),
}
fig7_network_order = ["SNOTEL", "SCAN", "USCRN", "SMOSMANIA", "OZNET", "ARM"]
fig7_networks = [network for network in fig7_network_order if network in fig7_df["network"].unique()]
fig7_markers = ["o", "s", "^", "D", "P", "X"]
fig7_colors = plt.cm.tab10(np.linspace(0, 1, len(fig7_networks)))
fig7_styles = {
    network: {"marker": fig7_markers[i % len(fig7_markers)], "color": fig7_colors[i]}
    for i, network in enumerate(fig7_networks)
}

fig7_plot_df = fig7_df.copy()
fig7_plot_df["validation_period_id"] = fig7_plot_df["window"].map(lambda value: fig7_validation_defs[value]["id"])
fig7_plot_df["validation_period"] = fig7_plot_df["window"].map(lambda value: fig7_validation_defs[value]["name"])
fig7_plot_df["validation_period_mapping"] = fig7_plot_df["window"].map(lambda value: fig7_validation_defs[value]["fine_periods"])
fig7_plot_df["positive_means"] = "DA better"
fig7_plot_df.to_csv(PAPER_FIG_DIR / "fig07_ismn_skill_by_validation_period_table.csv", index=False)

x = np.arange(len(fig7_window_order), dtype=float)
offsets = np.linspace(-0.24, 0.24, len(fig7_networks)) if len(fig7_networks) > 1 else np.array([0.0])

fig, axs = plt.subplots(2, 3, figsize=(13.6, 7.4))
legend_handles = []

for row_idx, domain in enumerate(fig7_domain_order):
    for col_idx, metric in enumerate(fig7_metric_order):
        ax = axs[row_idx, col_idx]
        lows = []
        highs = []

        for network_idx, network in enumerate(fig7_networks):
            sub = (
                fig7_plot_df[
                    (fig7_plot_df["network"] == network)
                    & (fig7_plot_df["domain"] == domain)
                    & (fig7_plot_df["metric"] == metric)
                ]
                .set_index("window")
                .reindex(fig7_window_order)
            )
            y = sub["delta_mean"].to_numpy(dtype=float)
            ci = sub["delta_ci_half"].to_numpy(dtype=float)
            xpos = x + offsets[network_idx]
            style = fig7_styles[network]
            handle = ax.errorbar(
                xpos,
                y,
                yerr=ci,
                fmt=style["marker"],
                linestyle="none",
                color=style["color"],
                ecolor="0.58",
                elinewidth=0.8,
                capsize=2.5,
                markersize=6.8,
                markeredgewidth=0.0,
                zorder=3,
            )
            if row_idx == 0 and col_idx == 0:
                legend_handles.append(handle.lines[0])
            valid = np.isfinite(y)
            if valid.any():
                ci0 = np.where(np.isfinite(ci), ci, 0.0)
                lows.extend((y[valid] - ci0[valid]).tolist())
                highs.extend((y[valid] + ci0[valid]).tolist())

        ax.axhline(0.0, color="0.15", linestyle=":", linewidth=0.9, zorder=1)
        ax.grid(axis="y", alpha=0.24, linewidth=0.7)
        ax.set_axisbelow(True)
        ax.set_xticks(x)
        ax.set_xticklabels([fig7_validation_defs[w]["axis_label"] for w in fig7_window_order], fontsize=8.0)
        ax.tick_params(axis="x", pad=2)
        ax.set_title(f"{fig7_domain_labels[domain]} | {fig7_metric_labels[metric]['title']}", fontsize=10.5)
        panel_label(ax, f"({chr(97 + row_idx * len(fig7_metric_order) + col_idx)})")
        ax.set_ylabel(fig7_metric_labels[metric]["ylabel"])

        y0, y1 = fig7_base_ylim[(domain, metric)]
        if lows and highs:
            data_min = float(np.nanmin(lows))
            data_max = float(np.nanmax(highs))
            span = max(data_max - data_min, y1 - y0, 1e-6)
            pad = 0.1 * span
            y0 = min(y0, data_min - pad)
            y1 = max(y1, data_max + pad)
        ax.set_ylim(y0, y1)


def fig7_network_count_label(network: str) -> str:
    subn = fig7_plot_df[fig7_plot_df["network"] == network]
    surf = (
        subn[(subn["metric"] == "R") & (subn["domain"] == "surface")]
        .set_index("window")
        .reindex(fig7_window_order)["n_pairs"]
        .to_numpy(dtype=float)
    )
    rz = (
        subn[(subn["metric"] == "R") & (subn["domain"] == "rz")]
        .set_index("window")
        .reindex(fig7_window_order)["n_pairs"]
        .to_numpy(dtype=float)
    )
    surf_valid = surf[np.isfinite(surf) & (surf > 0)]
    rz_valid = rz[np.isfinite(rz) & (rz > 0)]
    ns = int(np.nanmin(surf_valid)) if surf_valid.size else 0
    nr = int(np.nanmin(rz_valid)) if rz_valid.size else 0
    return f"{network} ({ns}/{nr})"

legend_labels = [fig7_network_count_label(network) for network in fig7_networks]
fig.legend(
    legend_handles,
    legend_labels,
    loc="lower center",
    ncol=len(fig7_networks),
    frameon=False,
    bbox_to_anchor=(0.5, 0.005),
    fontsize=8.5,
    handletextpad=0.35,
    columnspacing=0.8,
)
fig.suptitle("ISMN paired soil-moisture skill changes by validation period", fontsize=13)
fig.text(
    0.5,
    0.07,
    "Broader validation periods aggregate the P1–P9 observing-system periods. Network labels give minimum paired station counts (surface/root zone). Positive values indicate DA better; ubRMSE uses OL − DA.",
    ha="center",
    fontsize=8.2,
)
fig.subplots_adjust(left=0.065, right=0.995, top=0.90, bottom=0.20, wspace=0.30, hspace=0.36)

fig7_png = PAPER_FIG_DIR / "fig07_ismn_skill_by_validation_period.png"
fig7_pdf = PAPER_FIG_DIR / "fig07_ismn_skill_by_validation_period.pdf"
fig.savefig(fig7_png, dpi=300, bbox_inches="tight")
fig.savefig(fig7_pdf, bbox_inches="tight")
plt.show()

display(fig7_plot_df)
record_figure(
    "Fig. 7",
    fig7_png,
    sources=[str(fig7_table_path.relative_to(REPO_ROOT))],
    settings={
        "metric": "paired DA skill improvement; R/anomR = DA-OL, ubRMSE = OL-DA",
        "period_mapping": {defs["name"]: defs["fine_periods"] for defs in fig7_validation_defs.values()},
        "groups": fig7_networks,
        "format": "png",
        "dpi": 300,
    },
)
record_figure(
    "Fig. 7",
    fig7_pdf,
    sources=[str(fig7_table_path.relative_to(REPO_ROOT))],
    settings={
        "metric": "paired DA skill improvement; R/anomR = DA-OL, ubRMSE = OL-DA",
        "period_mapping": {defs["name"]: defs["fine_periods"] for defs in fig7_validation_defs.values()},
        "groups": fig7_networks,
        "format": "pdf",
    },
)
manifest = save_manifest()


## Figure 8: IMS Snow-Cover Skill and Terra/Aqua Scope Supplement

Main Figure 8 rebuilds the IMS categorical snow-cover skill maps from the precomputed per-cell count product. The supplemental panel uses the Discover rerun with custom P1/P2 scopes to compare the MODIS Terra-only and Terra+Aqua SCF periods.


In [ ]:
import matplotlib.colors as mcolors
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import xarray as xr

fig8_counts_path = p_rel("projects/IMS/output/ims_ol_da_cell_counts_metrics_SMAP_EASEv2_M36_GLOBAL_2000_2024_thr0p50_imsSnowDaysGe10.nc4")
fig8_table_path = p_rel("projects/IMS/output/ims_ol_da_comparison_table_SMAP_EASEv2_M36_GLOBAL_2000_2024_thr0p50_imsSnowDaysGe10.csv")
fig8_scope_path = p_rel("projects/IMS/output/ims_ol_da_scope_metadata_SMAP_EASEv2_M36_GLOBAL_2000_2024_thr0p50_imsSnowDaysGe10.csv")
fig8_ta_counts_path = p_rel("projects/IMS/output/ims_ol_da_cell_counts_metrics_SMAP_EASEv2_M36_GLOBAL_2000_2007_thr0p50_imsSnowDaysGe10_terraAquaScopes.nc4")
fig8_ta_table_path = p_rel("projects/IMS/output/ims_ol_da_comparison_table_SMAP_EASEv2_M36_GLOBAL_2000_2007_thr0p50_imsSnowDaysGe10_terraAquaScopes.csv")
fig8_ta_scope_path = p_rel("projects/IMS/output/ims_ol_da_scope_metadata_SMAP_EASEv2_M36_GLOBAL_2000_2007_thr0p50_imsSnowDaysGe10_terraAquaScopes.csv")
fig8_analysis_period_label = "calendar years 2000-2024"
fig8_analysis_period_short = "2000-2024"

fig8_metric_specs = [
    {"key": "accuracy", "label": "Accuracy", "higher_better": True},
    {"key": "hit_rate", "label": "Hit rate", "higher_better": True},
    {"key": "miss_rate", "label": "Miss rate", "higher_better": False},
    {"key": "false_alarm_ratio", "label": "False alarm ratio", "higher_better": False},
    {"key": "correct_rejection_rate", "label": "Correct rejection rate", "higher_better": True},
]
fig8_metric_labels = {spec["key"]: spec["label"] for spec in fig8_metric_specs}
fig8_metric_sign = {spec["key"]: (1.0 if spec["higher_better"] else -1.0) for spec in fig8_metric_specs}
fig8_metric_direction = {
    spec["key"]: ("DA - OL" if spec["higher_better"] else "OL - DA")
    for spec in fig8_metric_specs
}
fig8_metric_note = "; ".join(
    f"{spec['label']}: {fig8_metric_direction[spec['key']]}"
    for spec in fig8_metric_specs
)

fig8_table = pd.read_csv(fig8_table_path)
fig8_scope = pd.read_csv(fig8_scope_path)
fig8_ta_table = pd.read_csv(fig8_ta_table_path)
fig8_ta_scope = pd.read_csv(fig8_ta_scope_path)

with xr.open_dataset(fig8_counts_path) as fig8_ds:
    fig8_all_scope_idx = int(np.where(np.asarray(fig8_ds["scope_type_code"].values) == 0)[0][0])
    fig8_lon = np.asarray(fig8_ds["cell_lon"].values, dtype=float)
    fig8_lat = np.asarray(fig8_ds["cell_lat"].values, dtype=float)
    fig8_eligible = np.asarray(fig8_ds["cell_eligible"].values, dtype=bool)
    fig8_n_ol = np.asarray(fig8_ds["N"].isel(experiment=0, scope=fig8_all_scope_idx).values, dtype=float)
    fig8_n_da = np.asarray(fig8_ds["N"].isel(experiment=1, scope=fig8_all_scope_idx).values, dtype=float)
    fig8_n_common = np.minimum(fig8_n_ol, fig8_n_da)

    fig8_map_rows = []
    fig = plt.figure(figsize=(13.8, 6.2))
    gs = fig.add_gridspec(2, 3, left=0.035, right=0.985, top=0.86, bottom=0.17, wspace=0.08, hspace=0.20)
    cmap = plt.get_cmap("RdBu_r").copy()
    cmap.set_bad("#d4d4d4")
    norm = mcolors.TwoSlopeNorm(vmin=-0.25, vcenter=0.0, vmax=0.25)
    scatter_for_cbar = None

    for idx, spec in enumerate(fig8_metric_specs):
        ax = fig.add_subplot(gs[idx // 3, idx % 3], projection=ccrs.Robinson())
        ax.add_feature(cfeature.LAND, facecolor="#d8d8d8", edgecolor="none", zorder=0)
        ax.coastlines(resolution="50m", linewidth=0.35, color="0.35")
        ax.set_extent([-180, 180, 0, 90], crs=ccrs.PlateCarree())

        key = spec["key"]
        ol = np.asarray(fig8_ds[key].isel(experiment=0, scope=fig8_all_scope_idx).values, dtype=float)
        da = np.asarray(fig8_ds[key].isel(experiment=1, scope=fig8_all_scope_idx).values, dtype=float)
        improvement = fig8_metric_sign[key] * (da - ol)
        valid = fig8_eligible & np.isfinite(improvement) & np.isfinite(fig8_lon) & np.isfinite(fig8_lat) & (fig8_lat >= 0.0)
        domain = fig8_eligible & np.isfinite(fig8_lon) & np.isfinite(fig8_lat) & (fig8_lat >= 0.0)

        if domain.any():
            ax.scatter(
                fig8_lon[domain],
                fig8_lat[domain],
                s=0.45,
                c="#cfcfcf",
                marker="s",
                linewidths=0,
                alpha=0.75,
                transform=ccrs.PlateCarree(),
                rasterized=True,
                zorder=1,
            )
        if valid.any():
            scatter_for_cbar = ax.scatter(
                fig8_lon[valid],
                fig8_lat[valid],
                c=improvement[valid],
                s=0.45,
                cmap=cmap,
                norm=norm,
                marker="s",
                linewidths=0,
                transform=ccrs.PlateCarree(),
                rasterized=True,
                zorder=2,
            )
            mean_improvement = float(np.nanmean(improvement[valid]))
            median_improvement = float(np.nanmedian(improvement[valid]))
            improved_fraction = float(np.nanmean(improvement[valid] > 0.0) * 100.0)
            n_cells = int(valid.sum())
            n_pairs = int(np.nansum(fig8_n_common[valid]))
        else:
            mean_improvement = np.nan
            median_improvement = np.nan
            improved_fraction = np.nan
            n_cells = 0
            n_pairs = 0

        ax.set_title(f"     {spec['label']} improvement", fontsize=10.4, loc="left", pad=4)
        panel_label(ax, f"({chr(97 + idx)})", x=-0.055, y=1.04)
        ax.text(
            0.02,
            0.03,
            f"mean {mean_improvement:+.3f}\nimproved {improved_fraction:.0f}%",
            transform=ax.transAxes,
            ha="left",
            va="bottom",
            fontsize=8.0,
            bbox={"facecolor": "white", "edgecolor": "none", "alpha": 0.84, "pad": 2},
        )
        fig8_map_rows.append({
            "metric": key,
            "metric_label": spec["label"],
            "improvement_definition": fig8_metric_direction[key],
            "scope": "ALL_PERIOD",
            "season": "ALL",
            "analysis_period": fig8_analysis_period_short,
            "n_cells": n_cells,
            "n_pairs": n_pairs,
            "mean_improvement": mean_improvement,
            "median_improvement": median_improvement,
            "percent_cells_improved": improved_fraction,
        })

    ax_blank = fig.add_subplot(gs[1, 2])
    ax_blank.axis("off")
    ax_blank.text(
        0.02,
        0.96,
        "Period: 2000-2024\n"
        "Positive/red = DA better\n"
        "IMS snow/no-snow from daily categorical fields\n"
        "Model SCF threshold = 0.5\n"
        "Domain: cells with >=10 IMS snow days",
        ha="left",
        va="top",
        fontsize=9.0,
        linespacing=1.35,
    )

    if scatter_for_cbar is not None:
        cax = fig.add_axes([0.30, 0.075, 0.40, 0.035])
        cbar = fig.colorbar(scatter_for_cbar, cax=cax, orientation="horizontal")
        cbar.set_label("DA categorical skill improvement (fraction; red = improvement)")
        cbar.set_ticks([-0.25, -0.125, 0.0, 0.125, 0.25])

    fig.suptitle(f"IMS snow-cover categorical skill improvement, DA relative to OL ({fig8_analysis_period_label})", fontsize=13.0)

fig8_map_summary = pd.DataFrame(fig8_map_rows)
fig8_map_summary.to_csv(PAPER_FIG_DIR / "fig08_ims_snow_cover_skill_map_summary.csv", index=False)

fig8_png = PAPER_FIG_DIR / "fig08_ims_snow_cover_skill_maps.png"
fig8_pdf = PAPER_FIG_DIR / "fig08_ims_snow_cover_skill_maps.pdf"
fig.savefig(fig8_png, dpi=300, bbox_inches="tight")
fig.savefig(fig8_pdf, bbox_inches="tight")
plt.show()

def fig8_improvement_rows(table: pd.DataFrame, scopes: list[str]) -> pd.DataFrame:
    rows = []
    sub = table[
        table["scope"].isin(scopes)
        & table["season"].eq("ALL")
        & table["metric"].isin(fig8_metric_labels)
    ].copy()
    for _, row in sub.iterrows():
        key = row["metric"]
        sign = fig8_metric_sign[key]
        ci_vals = np.array([sign * row["delta_ci_lo"], sign * row["delta_ci_hi"]], dtype=float)
        rows.append({
            "scope": row["scope"],
            "metric": key,
            "metric_label": fig8_metric_labels[key],
            "improvement_definition": fig8_metric_direction[key],
            "ol": row["ol"],
            "da": row["da"],
            "improvement": sign * row["delta_da_minus_ol"],
            "improvement_ci_lo": float(np.nanmin(ci_vals)),
            "improvement_ci_hi": float(np.nanmax(ci_vals)),
            "n_pairs_ol": row["N_ol"],
            "n_pairs_da": row["N_da"],
        })
    return pd.DataFrame(rows)

fig8_scope_label = {
    "ALL_PERIOD": "All\n2000-2007",
    "P1_MODIS_Terra_SCF": "P1\nTerra",
    "P2_MODIS_Terra_Aqua_SCF": "P2\nTerra+Aqua",
}
fig8_scope_order = ["P1_MODIS_Terra_SCF", "P2_MODIS_Terra_Aqua_SCF", "ALL_PERIOD"]
fig8_ta_bars = fig8_improvement_rows(fig8_ta_table, fig8_scope_order)
fig8_ta_bars["scope_label"] = fig8_ta_bars["scope"].map(fig8_scope_label)
fig8_ta_bars.to_csv(PAPER_FIG_DIR / "fig08_supp_ims_terra_aqua_scope_bar_values.csv", index=False)

fig8_scope_period = {
    "P1_MODIS_Terra_SCF": "2000-06-01 to 2002-06-30",
    "P2_MODIS_Terra_Aqua_SCF": "2002-07-01 to 2007-05-31",
}
fig8_scope_row_label = {
    "P1_MODIS_Terra_SCF": "P1\nTerra SCF\n2000-06-01 to\n2002-06-30",
    "P2_MODIS_Terra_Aqua_SCF": "P2\nTerra+Aqua SCF\n2002-07-01 to\n2007-05-31",
}
fig8_map_scope_order = ["P1_MODIS_Terra_SCF", "P2_MODIS_Terra_Aqua_SCF"]
fig8_ta_map_rows = []

with xr.open_dataset(fig8_ta_counts_path) as fig8_ta_ds:
    fig8_ta_scope_names = np.asarray(fig8_ta_ds["scope_name"].values).astype(str)
    fig8_ta_scope_index = {name: int(np.where(fig8_ta_scope_names == name)[0][0]) for name in fig8_map_scope_order}
    fig8_ta_lon = np.asarray(fig8_ta_ds["cell_lon"].values, dtype=float)
    fig8_ta_lat = np.asarray(fig8_ta_ds["cell_lat"].values, dtype=float)
    fig8_ta_eligible = np.asarray(fig8_ta_ds["cell_eligible"].values, dtype=bool)

    fig = plt.figure(figsize=(15.0, 4.9))
    gs = fig.add_gridspec(
        len(fig8_map_scope_order),
        len(fig8_metric_specs),
        left=0.13,
        right=0.985,
        top=0.83,
        bottom=0.18,
        wspace=0.055,
        hspace=0.08,
    )
    cmap = plt.get_cmap("RdBu_r").copy()
    cmap.set_bad("#d4d4d4")
    norm = mcolors.TwoSlopeNorm(vmin=-0.25, vcenter=0.0, vmax=0.25)
    scatter_for_cbar = None

    for row_idx, scope_name in enumerate(fig8_map_scope_order):
        scope_idx = fig8_ta_scope_index[scope_name]
        n_ol = np.asarray(fig8_ta_ds["N"].isel(experiment=0, scope=scope_idx).values, dtype=float)
        n_da = np.asarray(fig8_ta_ds["N"].isel(experiment=1, scope=scope_idx).values, dtype=float)
        n_common = np.minimum(n_ol, n_da)
        domain = fig8_ta_eligible & np.isfinite(fig8_ta_lon) & np.isfinite(fig8_ta_lat) & (fig8_ta_lat >= 0.0)

        for col_idx, spec in enumerate(fig8_metric_specs):
            ax = fig.add_subplot(gs[row_idx, col_idx], projection=ccrs.Robinson())
            ax.add_feature(cfeature.LAND, facecolor="#d8d8d8", edgecolor="none", zorder=0)
            ax.coastlines(resolution="50m", linewidth=0.32, color="0.35")
            ax.set_extent([-180, 180, 0, 90], crs=ccrs.PlateCarree())

            key = spec["key"]
            ol = np.asarray(fig8_ta_ds[key].isel(experiment=0, scope=scope_idx).values, dtype=float)
            da = np.asarray(fig8_ta_ds[key].isel(experiment=1, scope=scope_idx).values, dtype=float)
            improvement = fig8_metric_sign[key] * (da - ol)
            valid = domain & np.isfinite(improvement)

            if domain.any():
                ax.scatter(
                    fig8_ta_lon[domain],
                    fig8_ta_lat[domain],
                    s=0.36,
                    c="#cfcfcf",
                    marker="s",
                    linewidths=0,
                    alpha=0.70,
                    transform=ccrs.PlateCarree(),
                    rasterized=True,
                    zorder=1,
                )
            if valid.any():
                scatter_for_cbar = ax.scatter(
                    fig8_ta_lon[valid],
                    fig8_ta_lat[valid],
                    c=improvement[valid],
                    s=0.36,
                    cmap=cmap,
                    norm=norm,
                    marker="s",
                    linewidths=0,
                    transform=ccrs.PlateCarree(),
                    rasterized=True,
                    zorder=2,
                )
                mean_improvement = float(np.nanmean(improvement[valid]))
                median_improvement = float(np.nanmedian(improvement[valid]))
                improved_fraction = float(np.nanmean(improvement[valid] > 0.0) * 100.0)
                n_cells = int(valid.sum())
                n_pairs = int(np.nansum(n_common[valid]))
            else:
                mean_improvement = np.nan
                median_improvement = np.nan
                improved_fraction = np.nan
                n_cells = 0
                n_pairs = 0

            if row_idx == 0:
                ax.set_title(f"     {spec['label']}", fontsize=9.5, loc="left", pad=3)
            panel_label(ax, f"({chr(97 + row_idx * len(fig8_metric_specs) + col_idx)})", x=-0.055, y=1.035)
            ax.text(
                0.02,
                0.03,
                f"mean {mean_improvement:+.3f}\nimproved {improved_fraction:.0f}%",
                transform=ax.transAxes,
                ha="left",
                va="bottom",
                fontsize=7.0,
                bbox={"facecolor": "white", "edgecolor": "none", "alpha": 0.84, "pad": 1.6},
            )
            fig8_ta_map_rows.append({
                "scope": scope_name,
                "period": fig8_scope_period[scope_name],
                "metric": key,
                "metric_label": spec["label"],
                "improvement_definition": fig8_metric_direction[key],
                "n_cells": n_cells,
                "n_pairs": n_pairs,
                "mean_improvement": mean_improvement,
                "median_improvement": median_improvement,
                "percent_cells_improved": improved_fraction,
            })

    fig.text(0.012, 0.62, fig8_scope_row_label["P1_MODIS_Terra_SCF"], ha="left", va="center", fontsize=8.0, fontweight="bold")
    fig.text(0.012, 0.35, fig8_scope_row_label["P2_MODIS_Terra_Aqua_SCF"], ha="left", va="center", fontsize=8.0, fontweight="bold")

    if scatter_for_cbar is not None:
        cax = fig.add_axes([0.30, 0.075, 0.40, 0.045])
        cbar = fig.colorbar(scatter_for_cbar, cax=cax, orientation="horizontal")
        cbar.set_label("DA categorical skill improvement (fraction; red = improvement)")
        cbar.set_ticks([-0.25, -0.125, 0.0, 0.125, 0.25])

    fig.suptitle("IMS snow-cover categorical skill improvement by MODIS SCF period", fontsize=12.8)
    fig.text(
        0.985,
        0.08,
        "Positive/red = DA better; domain: cells with >=10 IMS snow days.\nMiss rate and false alarm ratio use OL - DA; other metrics use DA - OL.",
        ha="right",
        va="bottom",
        fontsize=8.0,
    )

fig8_ta_map_summary = pd.DataFrame(fig8_ta_map_rows)
fig8_ta_map_summary.to_csv(PAPER_FIG_DIR / "fig08_supp_ims_terra_aqua_scope_map_summary.csv", index=False)

fig8_supp_maps_png = PAPER_FIG_DIR / "fig08_supp_ims_terra_aqua_scope_maps.png"
fig8_supp_maps_pdf = PAPER_FIG_DIR / "fig08_supp_ims_terra_aqua_scope_maps.pdf"
fig.savefig(fig8_supp_maps_png, dpi=300, bbox_inches="tight")
fig.savefig(fig8_supp_maps_pdf, bbox_inches="tight")
plt.show()

fig, axs = plt.subplots(1, len(fig8_metric_specs), figsize=(14.2, 4.0), sharey=False, constrained_layout=True)
bar_colors = {"P1_MODIS_Terra_SCF": "#4c78a8", "P2_MODIS_Terra_Aqua_SCF": "#72b7b2", "ALL_PERIOD": "#9aa0a6"}
x = np.arange(len(fig8_scope_order), dtype=float)

for idx, spec in enumerate(fig8_metric_specs):
    ax = axs[idx]
    key = spec["key"]
    sub = fig8_ta_bars[fig8_ta_bars["metric"].eq(key)].set_index("scope").reindex(fig8_scope_order)
    y = sub["improvement"].to_numpy(dtype=float)
    lo = sub["improvement_ci_lo"].to_numpy(dtype=float)
    hi = sub["improvement_ci_hi"].to_numpy(dtype=float)
    yerr = np.vstack([np.maximum(0.0, y - lo), np.maximum(0.0, hi - y)])
    ax.bar(
        x,
        y,
        yerr=yerr,
        color=[bar_colors[s] for s in fig8_scope_order],
        edgecolor="0.2",
        linewidth=0.8,
        capsize=2.5,
        error_kw={"elinewidth": 0.85, "ecolor": "0.25"},
    )
    ax.axhline(0.0, color="0.2", linestyle=":", linewidth=0.9)
    ax.grid(axis="y", color="0.86", linewidth=0.8)
    ax.set_axisbelow(True)
    ax.set_xticks(x)
    ax.set_xticklabels([fig8_scope_label[s] for s in fig8_scope_order], fontsize=8.0)
    ax.set_title(spec["label"], fontsize=10.0)
    panel_label(ax, f"({chr(97 + idx)})", x=-0.18, y=1.06)
    if idx == 0:
        ax.set_ylabel("DA improvement in score")
    max_abs = float(np.nanmax(np.abs(np.r_[lo, hi, y]))) if np.isfinite(np.r_[lo, hi, y]).any() else 0.05
    max_abs = max(max_abs * 1.25, 0.015)
    ax.set_ylim(-max_abs, max_abs)
    for xi, yi in zip(x, y):
        if np.isfinite(yi):
            va = "bottom" if yi >= 0 else "top"
            dy = 0.04 * max_abs if yi >= 0 else -0.04 * max_abs
            ax.text(xi, yi + dy, f"{yi:+.3f}", ha="center", va=va, fontsize=7.4)

fig.suptitle("IMS snow-cover skill improvement across MODIS Terra-only and Terra+Aqua SCF periods", fontsize=12.4)
fig.text(
    0.5,
    -0.045,
    "Positive values indicate DA better. P1 is MODIS Terra SCF; P2 is MODIS Terra+Aqua SCF. Miss rate and false alarm ratio are sign-flipped to OL - DA; other metrics use DA - OL. Error bars are cell-bootstrap 95% intervals.",
    ha="center",
    fontsize=8.2,
)

fig8_supp_png = PAPER_FIG_DIR / "fig08_supp_ims_terra_aqua_scope_bars.png"
fig8_supp_pdf = PAPER_FIG_DIR / "fig08_supp_ims_terra_aqua_scope_bars.pdf"
fig.savefig(fig8_supp_png, dpi=300, bbox_inches="tight")
fig.savefig(fig8_supp_pdf, bbox_inches="tight")
plt.show()

display(fig8_map_summary)
display(fig8_ta_map_summary)
display(fig8_ta_bars)

record_figure(
    "Fig. 8",
    fig8_png,
    sources=[str(fig8_counts_path.relative_to(REPO_ROOT)), str(fig8_table_path.relative_to(REPO_ROOT)), str(fig8_scope_path.relative_to(REPO_ROOT))],
    settings={
        "scope": f"ALL_PERIOD, ALL season, {fig8_analysis_period_label}",
        "metric": "categorical skill improvement; red/positive means DA better",
        "sign_convention": fig8_metric_direction,
        "domain": ">=10 IMS observed-snow days; northern hemisphere map extent",
        "format": "png",
        "dpi": 300,
    },
)
record_figure(
    "Fig. 8",
    fig8_pdf,
    sources=[str(fig8_counts_path.relative_to(REPO_ROOT)), str(fig8_table_path.relative_to(REPO_ROOT)), str(fig8_scope_path.relative_to(REPO_ROOT))],
    settings={
        "scope": f"ALL_PERIOD, ALL season, {fig8_analysis_period_label}",
        "metric": "categorical skill improvement; red/positive means DA better",
        "sign_convention": fig8_metric_direction,
        "domain": ">=10 IMS observed-snow days; northern hemisphere map extent",
        "format": "pdf",
    },
)
record_figure(
    "Fig. 8 supp.",
    fig8_supp_maps_png,
    sources=[str(fig8_ta_counts_path.relative_to(REPO_ROOT)), str(fig8_ta_scope_path.relative_to(REPO_ROOT))],
    settings={
        "scope": "P1 MODIS Terra SCF and P2 MODIS Terra+Aqua SCF maps",
        "metric": "categorical skill improvement; red/positive means DA better",
        "sign_convention": fig8_metric_direction,
        "domain": ">=10 IMS observed-snow days; northern hemisphere map extent",
        "format": "png",
        "dpi": 300,
    },
)
record_figure(
    "Fig. 8 supp.",
    fig8_supp_maps_pdf,
    sources=[str(fig8_ta_counts_path.relative_to(REPO_ROOT)), str(fig8_ta_scope_path.relative_to(REPO_ROOT))],
    settings={
        "scope": "P1 MODIS Terra SCF and P2 MODIS Terra+Aqua SCF maps",
        "metric": "categorical skill improvement; red/positive means DA better",
        "sign_convention": fig8_metric_direction,
        "domain": ">=10 IMS observed-snow days; northern hemisphere map extent",
        "format": "pdf",
    },
)
record_figure(
    "Fig. 8 supp.",
    fig8_supp_png,
    sources=[str(fig8_ta_table_path.relative_to(REPO_ROOT)), str(fig8_ta_scope_path.relative_to(REPO_ROOT))],
    settings={
        "scope": "P1 MODIS Terra SCF, P2 MODIS Terra+Aqua SCF, and all 2000-2007",
        "metric": "categorical skill improvement; red/positive means DA better",
        "sign_convention": fig8_metric_direction,
        "format": "png",
        "dpi": 300,
    },
)
record_figure(
    "Fig. 8 supp.",
    fig8_supp_pdf,
    sources=[str(fig8_ta_table_path.relative_to(REPO_ROOT)), str(fig8_ta_scope_path.relative_to(REPO_ROOT))],
    settings={
        "scope": "P1 MODIS Terra SCF, P2 MODIS Terra+Aqua SCF, and all 2000-2007",
        "metric": "categorical skill improvement; red/positive means DA better",
        "sign_convention": fig8_metric_direction,
        "format": "pdf",
    },
)
manifest = save_manifest()
display(manifest.tail(8))


## Figure 9: SNOTEL SWE Skill

Cached SNOTEL SWE station metrics and bootstrap bar summaries. The map row converts lower-is-better station metrics to an improvement convention: `OL - DA`, so positive/red values mean DA improved RMSE, ubRMSE, or absolute bias.


In [ ]:

import cartopy.crs as ccrs
import cartopy.feature as cfeature

fig9_station_path = p_rel("projects/SNOTEL/outputs_snotel_ol_da_validation/snotel_station_metrics_SMAP_EASEv2_M36_GLOBAL_20000601_20240601.csv")
fig9_bar_path = p_rel("projects/SNOTEL/outputs_snotel_ol_da_validation/tables/snotel_swe_toprow_bar_values_ci_SMAP_EASEv2_M36_GLOBAL_20000601_20240601.csv")

fig9_station = pd.read_csv(fig9_station_path)
fig9_bars = pd.read_csv(fig9_bar_path)

fig9_variable = "SNOMASLAND"
fig9_map_season = "ALL"
fig9_elev_thresh_m = 500.0
fig9_bar_seasons = ["ALL", "SON", "DJF", "MAM"]
fig9_metrics = ["rmse", "ubrmse", "bias"]
fig9_metric_labels = {"rmse": "RMSE", "ubrmse": "ubRMSE", "bias": "|Bias|"}
fig9_units = "kg m-2"
fig9_ol_color = "#2c6fbb"
fig9_da_color = "#e38d2c"

fig9_swe = fig9_station[fig9_station["variable"].eq(fig9_variable)].copy()
fig9_swe["abs_bias"] = fig9_swe["bias"].abs()

fig9_elev = (
    fig9_swe[(fig9_swe["season"].eq(fig9_map_season)) & (fig9_swe["experiment"].eq("OL"))]
    [["station", "elev_diff_m", "station_lat", "station_lon"]]
    .drop_duplicates("station")
    .copy()
)
fig9_elev["elev_diff_m"] = pd.to_numeric(fig9_elev["elev_diff_m"], errors="coerce")
fig9_station_keep = set(
    fig9_elev.loc[
        fig9_elev["elev_diff_m"].abs().lt(fig9_elev_thresh_m) & fig9_elev["elev_diff_m"].notna(),
        "station",
    ]
)


def build_fig9_improvement_table(metric: str) -> pd.DataFrame:
    value_col = "abs_bias" if metric == "bias" else metric
    sub = fig9_swe[
        fig9_swe["season"].eq(fig9_map_season)
        & fig9_swe["station"].isin(fig9_station_keep)
    ][["station", "experiment", value_col, "N", "station_lat", "station_lon", "elev_diff_m"]].copy()
    values = sub.pivot_table(index="station", columns="experiment", values=value_col, aggfunc="mean")
    counts = sub.pivot_table(index="station", columns="experiment", values="N", aggfunc="mean")
    meta = sub.groupby("station")[["station_lat", "station_lon", "elev_diff_m"]].first()
    out = meta.join(values, how="inner", rsuffix="_metric")
    for exp in ["OL", "DA"]:
        if exp not in out.columns:
            out[exp] = np.nan
        if exp not in counts.columns:
            counts[exp] = np.nan
    out["metric"] = metric
    out["metric_label"] = fig9_metric_labels[metric]
    out["ol_value"] = pd.to_numeric(out["OL"], errors="coerce")
    out["da_value"] = pd.to_numeric(out["DA"], errors="coerce")
    out["improvement_ol_minus_da"] = out["ol_value"] - out["da_value"]
    out["n_ol"] = counts["OL"]
    out["n_da"] = counts["DA"]
    out["da_improved"] = out["improvement_ol_minus_da"] > 0
    return out.reset_index()[[
        "station", "metric", "metric_label", "station_lat", "station_lon", "elev_diff_m",
        "ol_value", "da_value", "improvement_ol_minus_da", "da_improved", "n_ol", "n_da",
    ]]


fig9_improvement_table = pd.concat([build_fig9_improvement_table(metric) for metric in fig9_metrics], ignore_index=True)
fig9_improvement_table.to_csv(PAPER_FIG_DIR / "fig09_snotel_swe_station_improvement_table.csv", index=False)
fig9_bars.to_csv(PAPER_FIG_DIR / "fig09_snotel_swe_bar_values_ci.csv", index=False)

fig9_extent_sites = fig9_improvement_table[["station", "station_lat", "station_lon"]].drop_duplicates("station")
fig9_lon = pd.to_numeric(fig9_extent_sites["station_lon"], errors="coerce").to_numpy(dtype=float)
fig9_lat = pd.to_numeric(fig9_extent_sites["station_lat"], errors="coerce").to_numpy(dtype=float)
fig9_valid_xy = np.isfinite(fig9_lon) & np.isfinite(fig9_lat)
if fig9_valid_xy.any():
    lon_min, lon_max = np.nanmin(fig9_lon[fig9_valid_xy]), np.nanmax(fig9_lon[fig9_valid_xy])
    lat_min, lat_max = np.nanmin(fig9_lat[fig9_valid_xy]), np.nanmax(fig9_lat[fig9_valid_xy])
    pad_lon = max(1.0, 0.08 * max(lon_max - lon_min, 1.0))
    pad_lat = max(1.0, 0.08 * max(lat_max - lat_min, 1.0))
    fig9_extent = [
        max(-179.9, lon_min - pad_lon),
        min(179.9, lon_max + pad_lon),
        max(-89.9, lat_min - pad_lat),
        min(89.9, lat_max + pad_lat),
    ]
else:
    fig9_extent = [-170.0, -100.0, 25.0, 72.0]

fig = plt.figure(figsize=(14.2, 8.4), constrained_layout=True)
gs = fig.add_gridspec(2, 3, height_ratios=[0.9, 1.15])
bar_width = 0.36
x = np.arange(len(fig9_bar_seasons), dtype=float)

for col, metric in enumerate(fig9_metrics):
    ax = fig.add_subplot(gs[0, col])
    sub = fig9_bars[fig9_bars["metric"].eq(metric)].copy()
    means = sub.pivot_table(index="season", columns="experiment", values="mean", aggfunc="first").reindex(fig9_bar_seasons)
    lows = sub.pivot_table(index="season", columns="experiment", values="ci_low", aggfunc="first").reindex(fig9_bar_seasons)
    highs = sub.pivot_table(index="season", columns="experiment", values="ci_high", aggfunc="first").reindex(fig9_bar_seasons)
    counts = sub.pivot_table(index="season", columns="experiment", values="n_stations", aggfunc="first").reindex(fig9_bar_seasons)

    y_ol = means.get("OL", pd.Series(index=fig9_bar_seasons, dtype=float)).to_numpy(dtype=float)
    y_da = means.get("DA", pd.Series(index=fig9_bar_seasons, dtype=float)).to_numpy(dtype=float)
    ol_lo = lows.get("OL", pd.Series(index=fig9_bar_seasons, dtype=float)).to_numpy(dtype=float)
    ol_hi = highs.get("OL", pd.Series(index=fig9_bar_seasons, dtype=float)).to_numpy(dtype=float)
    da_lo = lows.get("DA", pd.Series(index=fig9_bar_seasons, dtype=float)).to_numpy(dtype=float)
    da_hi = highs.get("DA", pd.Series(index=fig9_bar_seasons, dtype=float)).to_numpy(dtype=float)
    yerr_ol = np.vstack([np.maximum(0, y_ol - ol_lo), np.maximum(0, ol_hi - y_ol)])
    yerr_da = np.vstack([np.maximum(0, y_da - da_lo), np.maximum(0, da_hi - y_da)])

    ax.bar(x - bar_width / 2, y_ol, width=bar_width, color=fig9_ol_color, yerr=yerr_ol, capsize=2.2, error_kw={"elinewidth": 0.8, "ecolor": "0.25"}, label="OL")
    ax.bar(x + bar_width / 2, y_da, width=bar_width, color=fig9_da_color, yerr=yerr_da, capsize=2.2, error_kw={"elinewidth": 0.8, "ecolor": "0.25"}, label="DA")
    all_improvement = float(means.loc["ALL", "OL"] - means.loc["ALL", "DA"]) if {"OL", "DA"}.issubset(means.columns) else np.nan
    ax.text(
        0.03,
        0.92,
        f"ALL improvement: {all_improvement:+.1f}",
        transform=ax.transAxes,
        fontsize=8.0,
        ha="left",
        va="top",
        bbox={"facecolor": "white", "edgecolor": "0.8", "alpha": 0.86, "pad": 2},
    )
    ax.set_title(f"{fig9_metric_labels[metric]} ({fig9_units})", fontsize=10.5)
    ax.set_xticks(x)
    ax.set_xticklabels(fig9_bar_seasons)
    ax.grid(axis="y", color="0.86", linewidth=0.8)
    ax.set_axisbelow(True)
    if col == 0:
        ax.set_ylabel("Station-mean SWE metric")
        ax.legend(loc="upper right", frameon=False)
    n_vals = counts.get("OL", pd.Series(index=fig9_bar_seasons, dtype=float)).to_numpy(dtype=float)
    for xi, n_val in zip(x, n_vals):
        if np.isfinite(n_val):
            ax.text(xi, -0.17, f"n={int(n_val)}", transform=ax.get_xaxis_transform(), ha="center", va="top", fontsize=7.1, color="0.35")
    panel_label(ax, f"({chr(97 + col)})", x=-0.11, y=1.08)

for col, metric in enumerate(fig9_metrics):
    ax = fig.add_subplot(gs[1, col], projection=ccrs.PlateCarree())
    ax.add_feature(cfeature.LAND, facecolor="0.94", edgecolor="none", zorder=0)
    ax.coastlines(resolution="50m", linewidth=0.45, color="0.45")
    ax.set_extent(fig9_extent, crs=ccrs.PlateCarree())
    sub = fig9_improvement_table[fig9_improvement_table["metric"].eq(metric)].copy()
    lon = pd.to_numeric(sub["station_lon"], errors="coerce").to_numpy(dtype=float)
    lat = pd.to_numeric(sub["station_lat"], errors="coerce").to_numpy(dtype=float)
    vals = pd.to_numeric(sub["improvement_ol_minus_da"], errors="coerce").to_numpy(dtype=float)
    valid = np.isfinite(lon) & np.isfinite(lat) & np.isfinite(vals)
    if valid.any():
        vmax = float(np.nanpercentile(np.abs(vals[valid]), 95))
        if not np.isfinite(vmax) or vmax <= 0:
            vmax = float(np.nanmax(np.abs(vals[valid]))) if valid.any() else 1.0
        vmax = max(vmax, 1.0)
        sc = ax.scatter(
            lon[valid],
            lat[valid],
            c=vals[valid],
            s=14,
            cmap="RdBu_r",
            vmin=-vmax,
            vmax=vmax,
            edgecolors="0.15",
            linewidths=0.18,
            alpha=0.92,
            transform=ccrs.PlateCarree(),
            zorder=2,
        )
        cbar = fig.colorbar(sc, ax=ax, orientation="horizontal", fraction=0.055, pad=0.035)
        cbar.set_label(f"Improvement, OL - DA ({fig9_units})", fontsize=8.0)
        mean_improvement = float(np.nanmean(vals[valid]))
        frac_improved = float(np.nanmean(vals[valid] > 0) * 100.0)
        ax.text(
            0.02,
            0.03,
            f"mean {mean_improvement:+.1f}; improved {frac_improved:.0f}%",
            transform=ax.transAxes,
            fontsize=7.6,
            ha="left",
            va="bottom",
            bbox={"facecolor": "white", "edgecolor": "none", "alpha": 0.86, "pad": 2},
        )
    ax.set_title(f"{fig9_metric_labels[metric]} improvement | ALL, |dz| < {int(fig9_elev_thresh_m)} m", fontsize=9.6)
    panel_label(ax, f"({chr(100 + col)})", x=-0.11, y=1.04)

fig.suptitle("SNOTEL SWE validation: station metrics and DA improvement", fontsize=13.0)

fig9_png = PAPER_FIG_DIR / "fig09_snotel_swe_skill.png"
fig9_pdf = PAPER_FIG_DIR / "fig09_snotel_swe_skill.pdf"
fig.savefig(fig9_png, dpi=300, bbox_inches="tight")
fig.savefig(fig9_pdf, bbox_inches="tight")
plt.show()

fig9_metric_summary = (
    fig9_improvement_table.groupby(["metric", "metric_label"], as_index=False)
    .agg(
        n_stations=("improvement_ol_minus_da", lambda x: int(np.isfinite(x).sum())),
        mean_improvement_ol_minus_da=("improvement_ol_minus_da", "mean"),
        median_improvement_ol_minus_da=("improvement_ol_minus_da", "median"),
        percent_improved=("improvement_ol_minus_da", lambda x: float(np.nanmean(np.asarray(x, dtype=float) > 0) * 100.0)),
    )
)
fig9_metric_summary.to_csv(PAPER_FIG_DIR / "fig09_snotel_swe_station_improvement_summary.csv", index=False)
display(fig9_metric_summary)

record_figure(
    "Fig. 9",
    fig9_png,
    sources=[fig9_station_path.name, fig9_bar_path.name],
    settings={
        "metrics": fig9_metrics,
        "bar_seasons": fig9_bar_seasons,
        "map_season": fig9_map_season,
        "map_metric": "OL - DA; positive/red means DA improved",
        "elevation_filter_m": fig9_elev_thresh_m,
        "format": "png",
        "dpi": 300,
    },
)
record_figure(
    "Fig. 9",
    fig9_pdf,
    sources=[fig9_station_path.name, fig9_bar_path.name],
    settings={
        "metrics": fig9_metrics,
        "bar_seasons": fig9_bar_seasons,
        "map_season": fig9_map_season,
        "map_metric": "OL - DA; positive/red means DA improved",
        "elevation_filter_m": fig9_elev_thresh_m,
        "format": "pdf",
    },
)
manifest = save_manifest()
display(manifest.tail(8))


## Figure 10: GHCN Snow-Depth Skill

Cached GHCN-Daily snow-depth station metrics. The map row uses the same lower-is-better improvement convention as Figure 9: `OL - DA`, so positive/red values mean DA improved RMSE, ubRMSE, or absolute bias.


In [ ]:
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from matplotlib.colors import TwoSlopeNorm

fig10_station_path = p_rel("projects/GHCN_snwd/outputs_ghcn_snwd_ol_da_validation/ghcn_station_metrics_baseline_core_SMAP_EASEv2_M36_GLOBAL_20000101_20241231.csv")
fig10_station = pd.read_csv(fig10_station_path)

fig10_variable = "SNODPLAND"
fig10_map_season = "ALL"
fig10_bar_seasons = ["ALL", "SON", "DJF", "MAM"]
fig10_metrics = ["rmse", "ubrmse", "abs_bias"]
fig10_metric_labels = {"rmse": "RMSE", "ubrmse": "ubRMSE", "abs_bias": "|Bias|"}
fig10_units = "mm"
fig10_ol_color = "#2c6fbb"
fig10_da_color = "#e38d2c"
fig10_bootstrap_n = 1000
fig10_rng = np.random.default_rng(20260412)

fig10_snwd = fig10_station[
    fig10_station["variable"].astype(str).eq(fig10_variable)
    & fig10_station["experiment"].astype(str).isin(["OL", "DA"])
].copy()
for col in ["N", "station_lat", "station_lon", "elev_diff_m", *fig10_metrics]:
    fig10_snwd[col] = pd.to_numeric(fig10_snwd[col], errors="coerce")


def fig10_bootstrap_mean_ci(values, n_boot=fig10_bootstrap_n):
    vals = np.asarray(values, dtype=float)
    vals = vals[np.isfinite(vals)]
    n = int(vals.size)
    if n == 0:
        return np.nan, np.nan, np.nan, 0
    if n == 1:
        return float(vals[0]), float(vals[0]), float(vals[0]), 1
    boot_means = np.empty(n_boot, dtype=float)
    for i in range(n_boot):
        boot_means[i] = float(fig10_rng.choice(vals, size=n, replace=True).mean())
    return (
        float(vals.mean()),
        float(np.nanpercentile(boot_means, 2.5)),
        float(np.nanpercentile(boot_means, 97.5)),
        n,
    )


fig10_bar_rows = []
for season in fig10_bar_seasons:
    for metric in fig10_metrics:
        for experiment in ["OL", "DA"]:
            sub = fig10_snwd[
                fig10_snwd["season"].astype(str).eq(season)
                & fig10_snwd["experiment"].astype(str).eq(experiment)
            ]
            mean, ci_low, ci_high, n_station = fig10_bootstrap_mean_ci(sub[metric].to_numpy(dtype=float))
            fig10_bar_rows.append({
                "season": season,
                "metric": metric,
                "experiment": experiment,
                "mean": mean,
                "ci_low": ci_low,
                "ci_high": ci_high,
                "n_stations": n_station,
                "bootstrap_resamples": fig10_bootstrap_n,
            })
fig10_bars = pd.DataFrame(fig10_bar_rows)
fig10_bars.to_csv(PAPER_FIG_DIR / "fig10_ghcn_snow_depth_bar_values_ci.csv", index=False)


def build_fig10_improvement_table(metric: str) -> pd.DataFrame:
    sub = fig10_snwd[
        fig10_snwd["season"].astype(str).eq(fig10_map_season)
    ][["station", "experiment", metric, "N", "station_lat", "station_lon", "elev_diff_m", "distance_km"]].copy()
    values = sub.pivot_table(index="station", columns="experiment", values=metric, aggfunc="mean")
    counts = sub.pivot_table(index="station", columns="experiment", values="N", aggfunc="mean")
    meta = sub.groupby("station")[["station_lat", "station_lon", "elev_diff_m", "distance_km"]].first()
    out = meta.join(values, how="inner", rsuffix="_metric")
    for exp in ["OL", "DA"]:
        if exp not in out.columns:
            out[exp] = np.nan
        if exp not in counts.columns:
            counts[exp] = np.nan
    out["metric"] = metric
    out["metric_label"] = fig10_metric_labels[metric]
    out["ol_value"] = pd.to_numeric(out["OL"], errors="coerce")
    out["da_value"] = pd.to_numeric(out["DA"], errors="coerce")
    out["improvement_ol_minus_da"] = out["ol_value"] - out["da_value"]
    out["n_ol"] = counts["OL"]
    out["n_da"] = counts["DA"]
    out["da_improved"] = out["improvement_ol_minus_da"] > 0
    return out.reset_index()[[
        "station", "metric", "metric_label", "station_lat", "station_lon", "elev_diff_m", "distance_km",
        "ol_value", "da_value", "improvement_ol_minus_da", "da_improved", "n_ol", "n_da",
    ]]


fig10_improvement_table = pd.concat([build_fig10_improvement_table(metric) for metric in fig10_metrics], ignore_index=True)
fig10_improvement_table = fig10_improvement_table[
    pd.to_numeric(fig10_improvement_table["station_lat"], errors="coerce").ge(0.0)
].copy()
fig10_improvement_table.to_csv(PAPER_FIG_DIR / "fig10_ghcn_snow_depth_station_improvement_table.csv", index=False)

fig = plt.figure(figsize=(15.8, 6.7), constrained_layout=False)
gs = fig.add_gridspec(2, 3, height_ratios=[1.0, 0.9], hspace=0.34, wspace=0.13)
fig.subplots_adjust(left=0.045, right=0.995, top=0.90, bottom=0.085)
bar_width = 0.36
x = np.arange(len(fig10_bar_seasons), dtype=float)

for col, metric in enumerate(fig10_metrics):
    ax = fig.add_subplot(gs[0, col])
    sub = fig10_bars[fig10_bars["metric"].eq(metric)].copy()
    means = sub.pivot_table(index="season", columns="experiment", values="mean", aggfunc="first").reindex(fig10_bar_seasons)
    lows = sub.pivot_table(index="season", columns="experiment", values="ci_low", aggfunc="first").reindex(fig10_bar_seasons)
    highs = sub.pivot_table(index="season", columns="experiment", values="ci_high", aggfunc="first").reindex(fig10_bar_seasons)
    counts = sub.pivot_table(index="season", columns="experiment", values="n_stations", aggfunc="first").reindex(fig10_bar_seasons)

    y_ol = means.get("OL", pd.Series(index=fig10_bar_seasons, dtype=float)).to_numpy(dtype=float)
    y_da = means.get("DA", pd.Series(index=fig10_bar_seasons, dtype=float)).to_numpy(dtype=float)
    ol_lo = lows.get("OL", pd.Series(index=fig10_bar_seasons, dtype=float)).to_numpy(dtype=float)
    ol_hi = highs.get("OL", pd.Series(index=fig10_bar_seasons, dtype=float)).to_numpy(dtype=float)
    da_lo = lows.get("DA", pd.Series(index=fig10_bar_seasons, dtype=float)).to_numpy(dtype=float)
    da_hi = highs.get("DA", pd.Series(index=fig10_bar_seasons, dtype=float)).to_numpy(dtype=float)
    yerr_ol = np.vstack([np.maximum(0, y_ol - ol_lo), np.maximum(0, ol_hi - y_ol)])
    yerr_da = np.vstack([np.maximum(0, y_da - da_lo), np.maximum(0, da_hi - y_da)])

    ax.bar(x - bar_width / 2, y_ol, width=bar_width, color=fig10_ol_color, yerr=yerr_ol, capsize=2.2, error_kw={"elinewidth": 0.8, "ecolor": "0.25"}, label="OL")
    ax.bar(x + bar_width / 2, y_da, width=bar_width, color=fig10_da_color, yerr=yerr_da, capsize=2.2, error_kw={"elinewidth": 0.8, "ecolor": "0.25"}, label="DA")
    all_improvement = float(means.loc["ALL", "OL"] - means.loc["ALL", "DA"]) if {"OL", "DA"}.issubset(means.columns) else np.nan
    ax.text(
        0.03,
        0.92,
        f"ALL improvement: {all_improvement:+.1f}",
        transform=ax.transAxes,
        fontsize=8.0,
        ha="left",
        va="top",
        bbox={"facecolor": "white", "edgecolor": "0.8", "alpha": 0.86, "pad": 2},
    )
    ax.set_title(f"{fig10_metric_labels[metric]} ({fig10_units})", fontsize=10.5)
    ax.set_xticks(x)
    ax.set_xticklabels(fig10_bar_seasons)
    ax.grid(axis="y", color="0.86", linewidth=0.8)
    ax.set_axisbelow(True)
    if col == 0:
        ax.set_ylabel("Station-mean snow-depth metric")
        ax.legend(loc="upper right", frameon=False)
    n_vals = counts.get("OL", pd.Series(index=fig10_bar_seasons, dtype=float)).to_numpy(dtype=float)
    for xi, n_val in zip(x, n_vals):
        if np.isfinite(n_val):
            ax.text(xi, -0.17, f"n={int(n_val)}", transform=ax.get_xaxis_transform(), ha="center", va="top", fontsize=7.1, color="0.35")
    panel_label(ax, f"({chr(97 + col)})", x=-0.11, y=1.08)

for col, metric in enumerate(fig10_metrics):
    ax = fig.add_subplot(gs[1, col], projection=ccrs.Robinson())
    ax.add_feature(cfeature.LAND, facecolor="0.94", edgecolor="none", zorder=0)
    ax.add_feature(cfeature.COASTLINE, linewidth=0.45, edgecolor="0.45", zorder=2)
    ax.add_feature(cfeature.BORDERS, linewidth=0.3, linestyle=":", edgecolor="0.55", zorder=2)
    ax.set_extent([-180, 180, 0, 90], crs=ccrs.PlateCarree())

    sub = fig10_improvement_table[fig10_improvement_table["metric"].eq(metric)].copy()
    lon = pd.to_numeric(sub["station_lon"], errors="coerce").to_numpy(dtype=float)
    lat = pd.to_numeric(sub["station_lat"], errors="coerce").to_numpy(dtype=float)
    vals = pd.to_numeric(sub["improvement_ol_minus_da"], errors="coerce").to_numpy(dtype=float)
    valid = np.isfinite(lon) & np.isfinite(lat) & np.isfinite(vals)
    if valid.any():
        vmax = float(np.nanpercentile(np.abs(vals[valid]), 95))
        if not np.isfinite(vmax) or vmax <= 0:
            vmax = float(np.nanmax(np.abs(vals[valid]))) if valid.any() else 1.0
        vmax = max(vmax, 1.0)
        norm = TwoSlopeNorm(vmin=-vmax, vcenter=0.0, vmax=vmax)
        sc = ax.scatter(
            lon[valid],
            lat[valid],
            c=vals[valid],
            s=8,
            cmap="RdBu_r",
            norm=norm,
            edgecolors="none",
            alpha=0.82,
            transform=ccrs.PlateCarree(),
            zorder=3,
        )
        cbar = fig.colorbar(sc, ax=ax, orientation="horizontal", fraction=0.055, pad=0.035)
        cbar.set_label(f"Improvement, OL - DA ({fig10_units})", fontsize=8.0)
        mean_improvement = float(np.nanmean(vals[valid]))
        frac_improved = float(np.nanmean(vals[valid] > 0) * 100.0)
        ax.text(
            0.02,
            0.03,
            f"mean {mean_improvement:+.1f}; improved {frac_improved:.0f}%",
            transform=ax.transAxes,
            fontsize=7.6,
            ha="left",
            va="bottom",
            bbox={"facecolor": "white", "edgecolor": "none", "alpha": 0.86, "pad": 2},
        )
    ax.set_title(f"{fig10_metric_labels[metric]} improvement | ALL season", fontsize=9.6)
    panel_label(ax, f"({chr(100 + col)})", x=-0.11, y=1.04)

fig.suptitle("GHCN snow-depth validation: station metrics and DA improvement", fontsize=13.0, y=0.975)

fig10_png = PAPER_FIG_DIR / "fig10_ghcn_snow_depth_skill.png"
fig10_pdf = PAPER_FIG_DIR / "fig10_ghcn_snow_depth_skill.pdf"
fig.savefig(fig10_png, dpi=300, bbox_inches="tight")
fig.savefig(fig10_pdf, bbox_inches="tight")
plt.show()

fig10_metric_summary = (
    fig10_improvement_table.groupby(["metric", "metric_label"], as_index=False)
    .agg(
        n_stations=("improvement_ol_minus_da", lambda x: int(np.isfinite(x).sum())),
        mean_improvement_ol_minus_da=("improvement_ol_minus_da", "mean"),
        median_improvement_ol_minus_da=("improvement_ol_minus_da", "median"),
        percent_improved=("improvement_ol_minus_da", lambda x: float(np.nanmean(np.asarray(x, dtype=float) > 0) * 100.0)),
    )
)
fig10_metric_summary.to_csv(PAPER_FIG_DIR / "fig10_ghcn_snow_depth_station_improvement_summary.csv", index=False)
display(fig10_metric_summary)

record_figure(
    "Fig. 10",
    fig10_png,
    sources=[fig10_station_path.name],
    settings={
        "metrics": fig10_metrics,
        "bar_seasons": fig10_bar_seasons,
        "map_season": fig10_map_season,
        "map_metric": "OL - DA; positive/red means DA improved",
        "baseline": "reported GHCN SNWD days; baseline-core station metrics",
        "format": "png",
        "dpi": 300,
    },
)
record_figure(
    "Fig. 10",
    fig10_pdf,
    sources=[fig10_station_path.name],
    settings={
        "metrics": fig10_metrics,
        "bar_seasons": fig10_bar_seasons,
        "map_season": fig10_map_season,
        "map_metric": "OL - DA; positive/red means DA improved",
        "baseline": "reported GHCN SNWD days; baseline-core station metrics",
        "format": "pdf",
    },
)
manifest = save_manifest()
display(manifest.tail(8))


## ERA5-Land Comparison Helpers

Shared loaders and plotting utilities for Figures 11-13. These figures use three broader validation-period aggregations of P1–P9, matching Figure 7, and the ERA5-Land strict periodized metric caches generated by the ERA5-Land workflow.


In [ ]:
import xarray as xr
from matplotlib.patches import Patch
from matplotlib.colors import TwoSlopeNorm

ERA5L_OL_SUMMARY = p_rel("projects/era5_land/notebooks/ERA5L_vs_OLv8_M36_strict_summary.nc")
ERA5L_DA_SUMMARY = p_rel("projects/era5_land/notebooks/ERA5L_vs_DAv8_M36_strict_summary.nc")
ERA5L_CACHE_DIRS = [
    p_rel("projects/era5_land/cache/era5l_periodized_metrics_bars"),
    p_rel("projects/era5_land/cache/era5_periodized_metrics"),
]
ERA5L_REQUIRED_VARS = {
    "R_sfc", "anomR_sfc", "ubRMSE_sfc", "R_rz", "anomR_rz", "ubRMSE_rz",
    "R_scf", "anomR_scf", "ubRMSE_scf", "rmse_scf", "bias_scf",
    "R_swe", "anomR_swe", "ubRMSE_swe", "rmse_swe", "bias_swe",
    "R_snwd", "anomR_snwd", "ubRMSE_snwd", "rmse_snwd", "bias_snwd",
}
ERA5L_PERIOD_LABELS = [
    "2000-06-01–2007-05-31",
    "2007-06-01–2015-03-31",
    "2015-04-01–2024-05-31",
    "2000-06-01–2024-05-31",
]
ERA5L_PERIOD_ALIAS = {
    "2000-06-01–2007-05-31": "V1",
    "2007-06-01–2015-03-31": "V2",
    "2015-04-01–2024-05-31": "V3",
    "2000-06-01–2024-05-31": "Full",
}
ERA5L_PERIOD_NAME = {
    "V1": "SCF-only period",
    "V2": "Pre-SMAP microwave",
    "V3": "SMAP-era microwave",
    "Full": "full period",
}
ERA5L_PERIOD_FINE = {
    "V1": "P1–P2",
    "V2": "P3–P5",
    "V3": "P6–P9",
    "Full": "P1-P9",
}
ERA5L_PERIOD_AXIS = {
    label: f"{ERA5L_PERIOD_NAME[ERA5L_PERIOD_ALIAS[label]]}\n{ERA5L_PERIOD_FINE[ERA5L_PERIOD_ALIAS[label]]}"
    for label in ERA5L_PERIOD_LABELS
}
ERA5L_COL_OL = "#2c6fbb"
ERA5L_COL_DA = "#e38d2c"
ERA5L_COL_OL_RZ = "#87b7e5"
ERA5L_COL_DA_RZ = "#f3bc73"
ERA5L_MIN_PAIRS = 24
ERA5L_SM_MASK_NOTE = "warm, snow-free mask: soil temperature > 275.15 K and SCF < 0.01"
ERA5L_REFERENCE_NOTE = "ERA5-Land is used as the higher-resolution land reanalysis reference; ERA5 products remain available from the strict workflow."


def _era5l_cache_ok(ds: xr.Dataset, required_vars: set[str]) -> bool:
    if not required_vars.issubset(set(ds.data_vars)):
        return False
    if "period" not in ds.coords:
        return False
    if [str(value) for value in ds["period"].values] != ERA5L_PERIOD_LABELS:
        return False
    if "lat" not in ds.coords or "lon" not in ds.coords:
        return False
    return True


def load_era5l_periodized_metrics(summary_path: Path, required_vars: set[str] = ERA5L_REQUIRED_VARS) -> xr.Dataset:
    stem = summary_path.stem
    candidates = []
    for cache_dir in ERA5L_CACHE_DIRS:
        candidates.extend(cache_dir.glob(f"{stem}__*.nc"))
    candidates = sorted(candidates, key=lambda p: p.stat().st_mtime, reverse=True)
    for candidate in candidates:
        ds = xr.open_dataset(candidate)
        if _era5l_cache_ok(ds, required_vars):
            ds.attrs["source_cache"] = str(candidate)
            return ds
        ds.close()
    raise FileNotFoundError(f"No compatible ERA5-Land periodized metric cache found for {summary_path.name}")


era5l_metrics = {
    "OL": load_era5l_periodized_metrics(ERA5L_OL_SUMMARY),
    "DA": load_era5l_periodized_metrics(ERA5L_DA_SUMMARY),
}
era5l_lat = era5l_metrics["DA"]["lat"].values
era5l_lon = era5l_metrics["DA"]["lon"].values
print("ERA5-Land metric caches:")
for exp, ds in era5l_metrics.items():
    print(f"  {exp}: {Path(ds.attrs['source_cache']).name}")


def mean_se_2d(da2d) -> tuple[float, float, int]:
    values = np.asarray(da2d.values if hasattr(da2d, "values") else da2d, dtype=float)
    valid = np.isfinite(values)
    n = int(valid.sum())
    if n == 0:
        return np.nan, np.nan, 0
    vals = values[valid]
    mean = float(np.nanmean(vals))
    se = float(np.nanstd(vals, ddof=1) / np.sqrt(n)) if n > 1 else np.nan
    return mean, se, n


def collect_era5l_stats(metric_key: str, periods: list[str]) -> pd.DataFrame:
    rows = []
    for period in periods:
        period_id = ERA5L_PERIOD_ALIAS[period]
        for exp, ds in era5l_metrics.items():
            mean, se, n = mean_se_2d(ds[metric_key].sel(period=period))
            rows.append({
                "period": period,
                "period_id": period_id,
                "period_name": ERA5L_PERIOD_NAME[period_id],
                "period_fine_mapping": ERA5L_PERIOD_FINE[period_id],
                "experiment": exp,
                "metric_key": metric_key,
                "mean": mean,
                "se": se,
                "n_cells": n,
            })
    return pd.DataFrame(rows)


def era5l_improvement_field(metric_key: str, period: str) -> xr.DataArray:
    if metric_key in {"R_sfc", "anomR_sfc", "R_rz", "anomR_rz"}:
        return era5l_metrics["DA"][metric_key].sel(period=period) - era5l_metrics["OL"][metric_key].sel(period=period)
    if metric_key in {"ubRMSE_sfc", "ubRMSE_rz", "rmse_scf", "ubRMSE_scf", "rmse_swe", "ubRMSE_swe", "rmse_snwd", "ubRMSE_snwd"}:
        return era5l_metrics["OL"][metric_key].sel(period=period) - era5l_metrics["DA"][metric_key].sel(period=period)
    raise ValueError(f"No standard improvement sign convention for {metric_key}")


def robust_symmetric_limit(arrays, percentile: float = 98.0, floor: float = 1e-6) -> float:
    vals = []
    for arr in arrays:
        data = np.asarray(arr, dtype=float)
        finite = data[np.isfinite(data)]
        if finite.size:
            vals.append(np.abs(finite))
    if not vals:
        return 1.0
    all_vals = np.concatenate(vals)
    limit = float(np.nanpercentile(all_vals, percentile))
    if not np.isfinite(limit) or limit <= floor:
        limit = float(np.nanmax(all_vals)) if all_vals.size else 1.0
    return max(limit, floor)


def weighted_mean_2d(values, weights=None) -> float:
    data = np.asarray(values, dtype=float)
    valid = np.isfinite(data)
    if weights is None:
        return float(np.nanmean(data)) if valid.any() else np.nan
    w = np.asarray(weights, dtype=float)
    valid &= np.isfinite(w) & (w > 0)
    if not valid.any():
        return np.nan
    return float(np.nansum(data[valid] * w[valid]) / np.nansum(w[valid]))


## Figure 11: ERA5-Land Soil-Moisture Bars

Surface and root-zone soil-moisture skill against ERA5-Land over the three broader validation-period aggregations of P1–P9 used in Figure 7.


In [ ]:
fig11_periods = ERA5L_PERIOD_LABELS[:3]
fig11_rows = [
    ("R", "R_sfc", "R_rz", "R"),
    ("Anomaly R", "anomR_sfc", "anomR_rz", "Anomaly R"),
    ("ubRMSE", "ubRMSE_sfc", "ubRMSE_rz", r"ubRMSE ($m^3$ $m^{-3}$)"),
]
fig11_stats = pd.concat(
    [collect_era5l_stats(metric, fig11_periods) for _, sfc_metric, rz_metric, _ in fig11_rows for metric in (sfc_metric, rz_metric)],
    ignore_index=True,
)
fig11_stats["domain"] = np.where(fig11_stats["metric_key"].str.endswith("_sfc"), "surface", "root-zone")
fig11_stats.to_csv(PAPER_FIG_DIR / "fig11_era5land_soil_moisture_bar_stats.csv", index=False)

fig, axs = plt.subplots(3, 3, figsize=(13.9, 8.7), constrained_layout=False)
fig.subplots_adjust(left=0.070, right=0.995, top=0.82, bottom=0.17, wspace=0.28, hspace=0.35)
legend_handles = [
    Patch(facecolor=ERA5L_COL_OL, edgecolor="0.2", label="surface OL"),
    Patch(facecolor=ERA5L_COL_DA, edgecolor="0.2", label="surface DA"),
    Patch(facecolor=ERA5L_COL_OL_RZ, edgecolor="0.2", label="root-zone OL"),
    Patch(facecolor=ERA5L_COL_DA_RZ, edgecolor="0.2", label="root-zone DA"),
]

for row_idx, (metric_title, sfc_key, rz_key, ylabel) in enumerate(fig11_rows):
    row_vals = []
    for metric_key in (sfc_key, rz_key):
        for period in fig11_periods:
            for exp in ("OL", "DA"):
                row_vals.append(mean_se_2d(era5l_metrics[exp][metric_key].sel(period=period))[:2])
    y_min = min([m - (0 if not np.isfinite(e) else e) for m, e in row_vals if np.isfinite(m)], default=0.0)
    y_max = max([m + (0 if not np.isfinite(e) else e) for m, e in row_vals if np.isfinite(m)], default=1.0)
    if metric_title in {"R", "Anomaly R"}:
        y_min = max(0.0, y_min)
        y_max = min(1.0, y_max)
    pad = 0.18 * max(y_max - y_min, 1e-6)

    for col_idx, period in enumerate(fig11_periods):
        ax = axs[row_idx, col_idx]
        stats = {}
        for domain, metric_key in (("surface", sfc_key), ("root-zone", rz_key)):
            for exp in ("OL", "DA"):
                stats[(domain, exp)] = mean_se_2d(era5l_metrics[exp][metric_key].sel(period=period))
        means = [stats[("surface", "OL")][0], stats[("surface", "DA")][0], stats[("root-zone", "OL")][0], stats[("root-zone", "DA")][0]]
        errs = [stats[("surface", "OL")][1], stats[("surface", "DA")][1], stats[("root-zone", "OL")][1], stats[("root-zone", "DA")][1]]
        ns = [stats[("surface", "OL")][2], stats[("surface", "DA")][2], stats[("root-zone", "OL")][2], stats[("root-zone", "DA")][2]]
        x = np.array([0.0, 0.70, 1.75, 2.45])
        bars = ax.bar(
            x,
            means,
            width=0.62,
            color=[ERA5L_COL_OL, ERA5L_COL_DA, ERA5L_COL_OL_RZ, ERA5L_COL_DA_RZ],
            edgecolor="0.2",
            linewidth=0.7,
            yerr=errs,
            capsize=3,
            error_kw={"elinewidth": 0.75, "ecolor": "0.25"},
        )
        ax.set_xticks(x)
        ax.set_xticklabels(["Sfc\nOL", "Sfc\nDA", "RZ\nOL", "RZ\nDA"], fontsize=8.0)
        ax.set_ylim(y_min - pad, y_max + pad)
        ax.grid(axis="y", color="0.87", linewidth=0.8)
        ax.set_axisbelow(True)
        ax.text(
            0.03,
            0.94,
            f"{metric_title}\nn={int(np.nanmin(ns))}",
            transform=ax.transAxes,
            ha="left",
            va="top",
            fontsize=8.1,
            bbox={"facecolor": "white", "edgecolor": "none", "alpha": 0.84, "pad": 1.8},
        )
        if row_idx == 0:
            pid = ERA5L_PERIOD_ALIAS[period]
            ax.set_title(ERA5L_PERIOD_AXIS[period], fontsize=9.2, pad=8)
        panel_label(ax, f"({chr(97 + row_idx * len(fig11_periods) + col_idx)})", x=-0.13, y=1.06)
        if col_idx == 0:
            ax.set_ylabel(ylabel)

fig.legend(handles=legend_handles, loc="lower center", bbox_to_anchor=(0.5, 0.085), ncol=4, frameon=False, fontsize=8.4)
fig.suptitle("ERA5-Land soil-moisture comparison by validation period", fontsize=13.0, y=0.965)
fig.text(
    0.5,
    0.040,
    f"{ERA5L_REFERENCE_NOTE} Statistics use {ERA5L_SM_MASK_NOTE}; min paired months = {ERA5L_MIN_PAIRS}.",
    ha="center",
    fontsize=8.1,
)
fig11_png = PAPER_FIG_DIR / "fig11_era5land_soil_moisture_bars.png"
fig11_pdf = PAPER_FIG_DIR / "fig11_era5land_soil_moisture_bars.pdf"
fig.savefig(fig11_png, dpi=300, bbox_inches="tight")
fig.savefig(fig11_pdf, bbox_inches="tight")
plt.show()

record_figure(
    "Fig. 11",
    fig11_png,
    sources=[Path(era5l_metrics["OL"].attrs["source_cache"]).name, Path(era5l_metrics["DA"].attrs["source_cache"]).name],
    settings={"periods": "broader validation-period aggregations: P1–P2, P3–P5, P6–P9", "reference": "ERA5-Land", "sm_mask": ERA5L_SM_MASK_NOTE, "format": "png", "dpi": 300},
)
record_figure(
    "Fig. 11",
    fig11_pdf,
    sources=[Path(era5l_metrics["OL"].attrs["source_cache"]).name, Path(era5l_metrics["DA"].attrs["source_cache"]).name],
    settings={"periods": "broader validation-period aggregations: P1–P2, P3–P5, P6–P9", "reference": "ERA5-Land", "sm_mask": ERA5L_SM_MASK_NOTE, "format": "pdf"},
)
manifest = save_manifest()
display(fig11_stats.head(12))


## Figure 12: ERA5-Land Soil-Moisture Improvement Maps

Surface and root-zone soil-moisture maps over the same three broader validation-period aggregations of P1–P9 as Figures 7 and 11. Map values are converted to a uniform improvement convention: positive/red means DA is better.


In [ ]:
fig12_periods = ERA5L_PERIOD_LABELS[:3]
fig12_rows = [
    ("Surface", "R", "R_sfc", "R improvement, DA - OL"),
    ("Root zone", "R", "R_rz", "R improvement, DA - OL"),
    ("Surface", "Anomaly R", "anomR_sfc", "Anomaly R improvement, DA - OL"),
    ("Root zone", "Anomaly R", "anomR_rz", "Anomaly R improvement, DA - OL"),
    ("Surface", "ubRMSE", "ubRMSE_sfc", r"ubRMSE improvement, OL - DA ($m^3$ $m^{-3}$)"),
    ("Root zone", "ubRMSE", "ubRMSE_rz", r"ubRMSE improvement, OL - DA ($m^3$ $m^{-3}$)"),
]
fig12_metric_groups = {
    "R": ["R_sfc", "R_rz"],
    "Anomaly R": ["anomR_sfc", "anomR_rz"],
    "ubRMSE": ["ubRMSE_sfc", "ubRMSE_rz"],
}
fig12_vlim = {
    metric_name: robust_symmetric_limit(
        [era5l_improvement_field(metric_key, period).values for metric_key in keys for period in fig12_periods],
        percentile=98.0,
        floor=0.002 if metric_name == "ubRMSE" else 0.02,
    )
    for metric_name, keys in fig12_metric_groups.items()
}
fig12_summary_rows = []

fig, axs = plt.subplots(
    len(fig12_rows),
    len(fig12_periods),
    figsize=(13.8, 12.4),
    subplot_kw={"projection": ccrs.Robinson()},
    constrained_layout=False,
)
plt.subplots_adjust(left=0.078, right=0.885, top=0.935, bottom=0.055, wspace=0.025, hspace=0.065)
fig12_mappables = {}
fig12_plot_lat = None
fig12_plot_lon = None

for row_idx, (domain_label, metric_label, metric_key, cbar_label) in enumerate(fig12_rows):
    for col_idx, period in enumerate(fig12_periods):
        ax = axs[row_idx, col_idx]
        values = era5l_improvement_field(metric_key, period).values
        plot_lat = lats2d[:values.shape[0], :values.shape[1]]
        plot_lon = lons2d[:values.shape[0], :values.shape[1]]
        values = np.where(plot_lat >= -60.0, values, np.nan)
        vlim = fig12_vlim[metric_label]
        mesh = ax.pcolormesh(
            plot_lon,
            plot_lat,
            values,
            transform=ccrs.PlateCarree(),
            cmap="RdBu_r",
            norm=TwoSlopeNorm(vmin=-vlim, vcenter=0.0, vmax=vlim),
            shading="auto",
            rasterized=True,
        )
        fig12_mappables[metric_label] = mesh
        add_map_base(ax)
        mean_val = weighted_mean_2d(values)
        n_cells = int(np.isfinite(values).sum())
        fig12_summary_rows.append({
            "domain": domain_label,
            "metric": metric_label,
            "metric_key": metric_key,
            "period": period,
            "period_id": ERA5L_PERIOD_ALIAS[period],
            "period_name": ERA5L_PERIOD_NAME[ERA5L_PERIOD_ALIAS[period]],
            "period_fine_mapping": ERA5L_PERIOD_FINE[ERA5L_PERIOD_ALIAS[period]],
            "mean_improvement": mean_val,
            "n_cells": n_cells,
            "positive_means": "DA better",
        })
        ax.text(
            0.02,
            0.04,
            f"mean {mean_val:+.3f}" if metric_label != "ubRMSE" else f"mean {mean_val:+.4f}",
            transform=ax.transAxes,
            fontsize=6.6,
            ha="left",
            va="bottom",
            bbox={"facecolor": "white", "edgecolor": "none", "alpha": 0.82, "pad": 1.5},
        )
        panel_label(ax, f"({chr(97 + row_idx * len(fig12_periods) + col_idx)})", x=-0.08, y=1.02)
        if row_idx == 0:
            pid = ERA5L_PERIOD_ALIAS[period]
            ax.set_title(ERA5L_PERIOD_AXIS[period], fontsize=8.5, pad=3)
        if col_idx == 0:
            ax.text(
                -0.09,
                0.5,
                f"{domain_label}\n{metric_label}",
                transform=ax.transAxes,
                ha="right",
                va="center",
                fontsize=8.0,
                weight="bold",
                linespacing=1.08,
            )

for metric_label, y0, y1 in [("R", 0.662, 0.915), ("Anomaly R", 0.382, 0.635), ("ubRMSE", 0.102, 0.355)]:
    cax = fig.add_axes([0.905, y0, 0.018, y1 - y0])
    cbar = fig.colorbar(fig12_mappables[metric_label], cax=cax, orientation="vertical")
    if metric_label == "ubRMSE":
        cbar.set_label(r"OL - DA ($m^3$ $m^{-3}$)", fontsize=7.4)
    else:
        cbar.set_label("DA - OL", fontsize=7.4)
    cbar.ax.tick_params(labelsize=6.8)

fig.suptitle("ERA5-Land soil-moisture DA improvement maps", fontsize=13.0, y=0.985)
fig.text(0.925, 0.95, "red = improvement", ha="center", fontsize=7.8, weight="bold")
fig12_png = PAPER_FIG_DIR / "fig12_era5land_soil_moisture_improvement_maps.png"
fig12_pdf = PAPER_FIG_DIR / "fig12_era5land_soil_moisture_improvement_maps.pdf"
fig.savefig(fig12_png, dpi=300, bbox_inches="tight")
fig.savefig(fig12_pdf, bbox_inches="tight")
plt.show()

fig12_summary = pd.DataFrame(fig12_summary_rows)
fig12_summary.to_csv(PAPER_FIG_DIR / "fig12_era5land_soil_moisture_map_summary.csv", index=False)
display(fig12_summary)
record_figure(
    "Fig. 12",
    fig12_png,
    sources=[Path(era5l_metrics["OL"].attrs["source_cache"]).name, Path(era5l_metrics["DA"].attrs["source_cache"]).name],
    settings={"periods": "broader validation-period aggregations: P1–P2, P3–P5, P6–P9", "metric_signs": "R/anomR = DA-OL; ubRMSE = OL-DA; positive/red = DA better", "format": "png", "dpi": 300},
)
record_figure(
    "Fig. 12",
    fig12_pdf,
    sources=[Path(era5l_metrics["OL"].attrs["source_cache"]).name, Path(era5l_metrics["DA"].attrs["source_cache"]).name],
    settings={"periods": "broader validation-period aggregations: P1–P2, P3–P5, P6–P9", "metric_signs": "R/anomR = DA-OL; ubRMSE = OL-DA; positive/red = DA better", "format": "pdf"},
)
manifest = save_manifest()


## Figure 13: ERA5-Land Snow Comparison

Full-period ERA5-Land snow comparison over the snow-possible domain. Bars use OL/DA labels and annotate each panel with DA improvement for lower-is-better snow metrics.


In [ ]:
fig13_period = ERA5L_PERIOD_LABELS[-1]
fig13_rows = [
    ("SCF", {"RMSE": ("rmse_scf", "RMSE (1)"), "ubRMSE": ("ubRMSE_scf", "ubRMSE (1)"), "Bias": ("bias_scf", "Bias (1)")}),
    ("SWE", {"RMSE": ("rmse_swe", "RMSE (m)"), "ubRMSE": ("ubRMSE_swe", "ubRMSE (m)"), "Bias": ("bias_swe", "Bias (m)")}),
    ("Snow depth", {"RMSE": ("rmse_snwd", "RMSE (m)"), "ubRMSE": ("ubRMSE_snwd", "ubRMSE (m)"), "Bias": ("bias_snwd", "Bias (m)")}),
]
fig13_cols = ["RMSE", "ubRMSE", "Bias"]
fig13_stats_rows = []

fig, axs = plt.subplots(3, 3, figsize=(12.0, 8.8), constrained_layout=False)
fig.subplots_adjust(left=0.075, right=0.995, top=0.88, bottom=0.14, wspace=0.28, hspace=0.43)

for row_idx, (snow_label, metric_map) in enumerate(fig13_rows):
    for col_idx, col_name in enumerate(fig13_cols):
        ax = axs[row_idx, col_idx]
        metric_key, ylabel = metric_map[col_name]
        stats = {exp: mean_se_2d(era5l_metrics[exp][metric_key].sel(period=fig13_period)) for exp in ("OL", "DA")}
        means = [stats["OL"][0], stats["DA"][0]]
        errs = [stats["OL"][1], stats["DA"][1]]
        ns = [stats["OL"][2], stats["DA"][2]]
        x = np.arange(2, dtype=float)
        ax.bar(
            x,
            means,
            width=0.66,
            color=[ERA5L_COL_OL, ERA5L_COL_DA],
            edgecolor="0.2",
            linewidth=0.7,
            yerr=errs,
            capsize=3,
            error_kw={"elinewidth": 0.75, "ecolor": "0.25"},
        )
        ax.set_xticks(x)
        ax.set_xticklabels(["OL", "DA"])
        ax.grid(axis="y", color="0.87", linewidth=0.8)
        ax.set_axisbelow(True)
        if col_name == "Bias":
            improvement = abs(means[0]) - abs(means[1])
            improvement_label = f"|bias| improvement: {improvement:+.3g}"
        else:
            improvement = means[0] - means[1]
            improvement_label = f"improvement: {improvement:+.3g}"
        y_low = min([m - (0 if not np.isfinite(e) else e) for m, e in zip(means, errs) if np.isfinite(m)], default=0.0)
        y_high = max([m + (0 if not np.isfinite(e) else e) for m, e in zip(means, errs) if np.isfinite(m)], default=1.0)
        if col_name == "Bias":
            y_low = min(y_low, 0.0)
            y_high = max(y_high, 0.0)
            ax.axhline(0, color="0.2", linestyle=":", linewidth=0.8)
        pad = 0.22 * max(y_high - y_low, 1e-8)
        ax.set_ylim(y_low - pad, y_high + pad)
        ax.text(
            0.03,
            0.94,
            improvement_label,
            transform=ax.transAxes,
            ha="left",
            va="top",
            fontsize=7.5,
            bbox={"facecolor": "white", "edgecolor": "0.8", "alpha": 0.86, "pad": 1.8},
        )
        ax.set_title(f"{snow_label} | {col_name}\nn={int(np.nanmin(ns))}", fontsize=9.0)
        panel_label(ax, f"({chr(97 + row_idx * len(fig13_cols) + col_idx)})", x=-0.13, y=1.06)
        if col_idx == 0:
            ax.set_ylabel(ylabel)
        for exp, mean, se, n in (("OL", means[0], errs[0], ns[0]), ("DA", means[1], errs[1], ns[1])):
            fig13_stats_rows.append({
                "snow_variable": snow_label,
                "metric": col_name,
                "metric_key": metric_key,
                "experiment": exp,
                "period": fig13_period,
                "period_id": "Full",
                "period_name": ERA5L_PERIOD_NAME["Full"],
                "period_fine_mapping": ERA5L_PERIOD_FINE["Full"],
                "mean": mean,
                "se": se,
                "n_cells": n,
                "improvement_annotation": improvement_label,
                "domain": "snow-possible",
            })

fig.suptitle("ERA5-Land snow comparison over the snow-possible domain", fontsize=13.0, y=0.965)
fig.text(
    0.5,
    0.055,
    "Full period: 2000-06 to 2024-05 (P1-P9). Snow-possible cells have snow in either OL/DA model output or ERA5-Land during at least one month.",
    ha="center",
    fontsize=8.2,
)
fig13_png = PAPER_FIG_DIR / "fig13_era5land_snow_comparison_bars.png"
fig13_pdf = PAPER_FIG_DIR / "fig13_era5land_snow_comparison_bars.pdf"
fig.savefig(fig13_png, dpi=300, bbox_inches="tight")
fig.savefig(fig13_pdf, bbox_inches="tight")
plt.show()

fig13_stats = pd.DataFrame(fig13_stats_rows)
fig13_stats.to_csv(PAPER_FIG_DIR / "fig13_era5land_snow_comparison_stats.csv", index=False)
display(fig13_stats.head(12))
record_figure(
    "Fig. 13",
    fig13_png,
    sources=[Path(era5l_metrics["OL"].attrs["source_cache"]).name, Path(era5l_metrics["DA"].attrs["source_cache"]).name],
    settings={"period": "Full P1-P9", "domain": "snow-possible", "labels": "OL/DA", "format": "png", "dpi": 300},
)
record_figure(
    "Fig. 13",
    fig13_pdf,
    sources=[Path(era5l_metrics["OL"].attrs["source_cache"]).name, Path(era5l_metrics["DA"].attrs["source_cache"]).name],
    settings={"period": "Full P1-P9", "domain": "snow-possible", "labels": "OL/DA", "format": "pdf"},
)
manifest = save_manifest()
